# STEP 4: All Models Training

**Purpose**: Train all models for CVE prioritization using a common temporal protocol.

**What this notebook does**:
1. **Data Preparation** - Load features and create temporal splits
2. **Reference Scoring Models** - CVSS-only and heuristic rankers
3. **Graph-Based Models** - DiffusionRank, RGCN, and ensemble variants
4. **Learning-to-Rank Training** - XGBoost + LambdaMART rankers
5. **Model Artifact Export** - Save trained models and prediction artifacts for STEP 5
6. **Training Diagnostics** - Review training quality and model readiness
7. **Explainability** - SHAP values for feature importance
8. **Handoff for Evaluation** - Prepare outputs for STEP 5 comparison notebook

**Key Innovation**: Confidence-weighted training
- Each CVE's label has a confidence score
- Training loss weighted by confidence
- High-confidence examples (KEV) have more influence
- Low-confidence examples contribute less to gradient

---



## 1. Setup & Imports

Initialize project environment, load data pipelines, and import training utilities.

In [1]:
import sys
import os
from pathlib import Path
import warnings
import json
import uuid
import socket
import platform
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import wilcoxon

warnings.filterwarnings('ignore')

# Setup project paths
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root))
os.chdir(project_root)

print(f"[OK] Project root: {project_root}")
print("[OK] Core imports successful")
print("[OK] Project/ML imports deferred to Cell 4")

[OK] Project root: /Users/vinayksharma/AirDnd/cti_recommender
[OK] Core imports successful
[OK] Project/ML imports deferred to Cell 4


In [2]:
# Project imports (separated from setup cell to avoid startup hangs)
from src.models.ltr import train_lambdarank, save_model, get_default_ltr_params
from src.models.baselines import compute_cvss_only_scores, compute_heuristic_scores
from src.utils.notebook_helpers import save_plot, save_dataframe, display_sample, setup_notebook_output, print_header, print_subheader, print_separator
from config.experiment_config import get_config

# Configure notebook display
setup_notebook_output()

# Load experiment configuration
exp_cfg = get_config()

# Config-driven temporal split ratios
split_cfg = exp_cfg.temporal_splits.percentage_split
TRAIN_RATIO = float(split_cfg.get('train', 0.70))
VAL_RATIO   = float(split_cfg.get('val',   0.15))
TEST_RATIO  = float(split_cfg.get('test',  0.15))

# Config-driven ranking evaluation cutoffs
EVAL_K_VALUES = sorted({int(k) for k in exp_cfg.evaluation.k_values if int(k) > 0})
if not EVAL_K_VALUES:
    EVAL_K_VALUES = [10, 20, 100]

# Config-driven year-based cutoff
year_split_cfg = exp_cfg.temporal_splits.year_split
test_years = sorted(int(y) for y in year_split_cfg.get('test_years', [2025]))
thesis_test_start_year = test_years[0] if test_years else 2025
THESIS_CUTOFF_DATE = pd.Timestamp(f'{thesis_test_start_year - 1}-12-31', tz='UTC')

print(f'[OK] Config profile: {exp_cfg._profile}')
print(f'[OK] Split ratios (train/val/test): {TRAIN_RATIO:.2f}/{VAL_RATIO:.2f}/{TEST_RATIO:.2f}')
print(f'[OK] Evaluation k-values: {EVAL_K_VALUES}')
print(f'[OK] Thesis cutoff date: {THESIS_CUTOFF_DATE.date()}')

# Traceability setup — implementation in src/utils/run_tracker.py
RUN_ID = f"step4_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"
TRACE_LOG_DIR = project_root / 'logs' / 'runs' / RUN_ID
TRACE_LOG_DIR.mkdir(parents=True, exist_ok=True)

from src.utils.run_tracker import RunTracker
_tracker = RunTracker(run_id=RUN_ID, log_dir=TRACE_LOG_DIR,
                      notebook='STEP_4_All_Models_Training.ipynb')

# Expose tracker state and add notebook-level manifest fields
_tracker.manifest.update({
    'config_profile': getattr(exp_cfg, '_profile', 'unknown'),
    'split_ratios': {'train': TRAIN_RATIO, 'val': VAL_RATIO, 'test': TEST_RATIO},
    'eval_k_values': EVAL_K_VALUES,
})
RUN_MANIFEST      = _tracker.manifest
RUN_MANIFEST_FILE = _tracker.manifest_file
TRACE_LOG_FILE    = _tracker.trace_file

# Module-level aliases so all downstream cells work unchanged
trace_event       = _tracker.trace_event
trace_stage_done  = _tracker.trace_stage_done
register_artifact = _tracker.register_artifact

trace_stage_done(
    'notebook_start',
    status='ok',
    profile=getattr(exp_cfg, '_profile', 'unknown'),
)
print(f'[OK] Trace run id: {RUN_ID}')
print(f'[OK] Trace log:    {TRACE_LOG_FILE}')
print(f'[OK] Run manifest: {RUN_MANIFEST_FILE}')

[OK] Notebook output configured
[OK] Config profile: production
[OK] Split ratios (train/val/test): 0.70/0.15/0.15
[OK] Evaluation k-values: [10, 20, 100]
[OK] Thesis cutoff date: 2024-12-31
[TRACE] notebook_start | ok | {'profile': 'production'}
[OK] Trace run id: step4_20260329T101003Z_d528f4d9
[OK] Trace log:    /Users/vinayksharma/AirDnd/cti_recommender/logs/runs/step4_20260329T101003Z_d528f4d9/trace_events.jsonl
[OK] Run manifest: /Users/vinayksharma/AirDnd/cti_recommender/logs/runs/step4_20260329T101003Z_d528f4d9/run_manifest.json


## 2. Load Processed Features

Read labeled CVE dataset from STEP_3 with all 53 features and confidence-weighted labels.

In [3]:
# Load features from Feature_Engineering notebook output
features_dir = project_root / 'outputs' / 'features'
latest_features = sorted(features_dir.glob('features_with_labels_*.csv'))[-1]

print(f"Loading features from: {latest_features.name}")
df = pd.read_csv(latest_features, low_memory=False)

# Fast datetime parse: ISO8601 format hint avoids per-row inference (210K rows)
df['published'] = pd.to_datetime(df['published'], format='ISO8601', utc=True)

# CVSS normalization (0-10 -> 0-1)
df['cvss_norm'] = df['cvss'] / 10.0

# Recency score (days since publication, normalized)
if 'modified' in df.columns:
    df['modified'] = pd.to_datetime(df['modified'], format='mixed', utc=True)
days_since_pub = (pd.Timestamp.now(tz='UTC') - df['published']).dt.days
max_days = days_since_pub.max()
df['recency_score'] = 1.0 - (days_since_pub / max_days)  # More recent = higher score

# has_attack flag (alias for attack_flag for baseline compatibility)
if 'attack_flag' in df.columns:
    df['has_attack'] = df['attack_flag']
else:
    df['has_attack'] = 0

# Encode categorical features to numeric (handle all edge cases)
for col in ['cvss_severity_category', 'cwe_category', 'curated_severity']:
    if col in df.columns:
        df[col] = df[col].astype(str).replace('nan', float('nan'))
        df[col] = df[col].fillna('unknown')
        df[col] = pd.Categorical(df[col]).codes

print_header('DATA LOADED')
print(f"Total CVEs: {len(df):,}")
print(f"Features: {len([c for c in df.columns if c not in ['cve_id', 'published', 'modified', 'soft_label', 'label_confidence']])}")
print(f"Date range: {df['published'].min().date()} to {df['published'].max().date()}")
print(f"Label range: {df['soft_label'].min()} to {df['soft_label'].max()}")
print(f"Mean confidence: {df['label_confidence'].mean():.3f}")
print_separator()

display_sample(df[['cve_id', 'published', 'cvss', 'cvss_norm', 'soft_label', 'label_confidence']], title="Data Summary ")

Loading features from: features_with_labels_20260329.csv

DATA LOADED

Total CVEs: 210,147
Features: 54
Date range: 2018-01-01 to 2025-12-31
Label range: 0 to 3
Mean confidence: 0.329



Showing 20 of 210,147 rows


,cve_id,published,cvss,cvss_norm,soft_label,label_confidence
0,CVE-2025-67711,2025-12-31 23:15:42.413000+00:00,6.1,0.61,1,0.3
1,CVE-2025-67710,2025-12-31 23:15:42.270000+00:00,6.1,0.61,1,0.3
2,CVE-2025-67709,2025-12-31 23:15:42.130000+00:00,6.1,0.61,1,0.3
3,CVE-2025-67708,2025-12-31 23:15:41.980000+00:00,6.1,0.61,1,0.3
4,CVE-2025-67707,2025-12-31 23:15:41.833000+00:00,5.6,0.56,1,0.3
5,CVE-2025-67706,2025-12-31 23:15:41.687000+00:00,5.6,0.56,1,0.3
6,CVE-2025-67705,2025-12-31 23:15:41.540000+00:00,6.1,0.61,1,0.3
7,CVE-2025-67704,2025-12-31 23:15:41.387000+00:00,6.1,0.61,1,0.3
8,CVE-2025-67703,2025-12-31 23:15:40.540000+00:00,6.1,0.61,1,0.3
9,CVE-2025-69288,2025-12-31 22:15:49.410000+00:00,9.1,0.91,1,0.3


... 210,127 more rows


## 3. Temporal Train/Validation/Test Splits

Critical for time-series data: train on past, validate on recent, test on future

In [4]:
from src.features.engineering import fit_categorical_mapping, apply_categorical_mapping
from src.utils.temporal import make_temporal_splits_flexible as _make_splits

# Implementations are in src/features/engineering.py and src/utils/temporal.py
train_df, val_df, test_df = _make_splits(
    df,
    config={'strategy': 'percentage', 'percentage_split':
            {'train': TRAIN_RATIO, 'val': VAL_RATIO, 'test': TEST_RATIO}},
    date_col='published',
)

CATEGORICAL_COLS = ['cvss_severity_category', 'cwe_category', 'curated_severity']
cat_mapping_main = fit_categorical_mapping(train_df, CATEGORICAL_COLS)
train_df = apply_categorical_mapping(train_df, cat_mapping_main)
val_df = apply_categorical_mapping(val_df, cat_mapping_main)
test_df = apply_categorical_mapping(test_df, cat_mapping_main)

print()
print('[DEBUG] Categorical column dtypes after train-fitted encoding:')
for col in CATEGORICAL_COLS:
    if col in train_df.columns:
        print(f"  {col}: {train_df[col].dtype} (train unique={train_df[col].nunique()})")

print_header('TEMPORAL SPLITS CREATED')
print('[STATS] Split Sizes:')
print(f"  Train: {len(train_df):,} CVEs ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Val:   {len(val_df):,} CVEs ({len(val_df)/len(df)*100:.1f}%)")
print(f"  Test:  {len(test_df):,} CVEs ({len(test_df)/len(df)*100:.1f}%)")

print()
print(' Date Ranges:')
print(f"  Train: {train_df['published'].min().date()} to {train_df['published'].max().date()}")
print(f"  Val:   {val_df['published'].min().date()} to {val_df['published'].max().date()}")
print(f"  Test:  {test_df['published'].min().date()} to {test_df['published'].max().date()}")

print()
print('  Label Distribution:')
for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    high_priority = (split_df['soft_label'] >= 2).mean() * 100
    print(f"  {split_name}: {high_priority:.1f}% high-priority (label>=2)")

print_separator()
print()

split_timeline = pd.DataFrame([
    {'Split': 'Train', 'Start': train_df['published'].min(), 'End': train_df['published'].max(), 'Count': len(train_df)},
    {'Split': 'Val', 'Start': val_df['published'].min(), 'End': val_df['published'].max(), 'Count': len(val_df)},
    {'Split': 'Test', 'Start': test_df['published'].min(), 'End': test_df['published'].max(), 'Count': len(test_df)}
])

fig = px.timeline(
    split_timeline,
    x_start='Start',
    x_end='End',
    y='Split',
    color='Count',
    title='Temporal Train/Val/Test Splits',
    labels={'Count': 'Number of CVEs'},
    text='Count'
)
fig.update_traces(texttemplate='%{text:,}', textposition='inside')
fig.update_layout(height=300)

save_plot(fig, 'temporal_splits')
print('[OK] Temporal split visualization saved')

Percentage-based Temporal Split:
  Train:       147,102 samples (70.0%)
  Validation:   31,522 samples (15.0%)
  Test:         31,523 samples (15.0%)
  Total:       210,147

[DEBUG] Categorical column dtypes after train-fitted encoding:
  cvss_severity_category: int64 (train unique=3)
  cwe_category: int64 (train unique=9)
  curated_severity: int64 (train unique=3)

TEMPORAL SPLITS CREATED

[STATS] Split Sizes:
  Train: 147,102 CVEs (70.0%)
  Val:   31,522 CVEs (15.0%)
  Test:  31,523 CVEs (15.0%)

 Date Ranges:
  Train: 2018-01-01 to 2024-07-09
  Val:   2024-07-09 to 2025-04-08
  Test:  2025-04-08 to 2025-12-31

  Label Distribution:
  Train: 3.4% high-priority (label>=2)
  Val: 1.8% high-priority (label>=2)
  Test: 0.7% high-priority (label>=2)


[OK] Temporal split visualization saved


## 4. Define Feature Columns

Specify feature subsets for training: basic enrichments, CVSS decomposition, CWE intelligence, NLP signals, and interaction features.

In [5]:
# Define all feature columns (53 total: 16 basic + 37 enhanced)
basic_features = [
    'kev_flag', 'epss_score', 'epss_percentile', 
    'is_healthcare', 'healthcare_score',
    'attack_flag', 'attack_technique_count',
    'chpl_flag', 'is_curated', 'curated_severity'
]

cvss_features = [
    'cvss_av', 'cvss_ac', 'cvss_pr', 'cvss_ui', 'cvss_s',
    'cvss_c', 'cvss_i', 'cvss_a', 'cvss_score_derived', 'cvss_severity_category'
]

cwe_features = [
    'cwe_is_top25', 'cwe_is_injection', 'cwe_is_crypto',
    'cwe_is_access_control', 'cwe_is_input_validation',
    'cwe_is_memory_corruption', 'cwe_category', 'cwe_severity_score'
]

nlp_features = [
    'desc_has_rce', 'desc_has_auth_bypass', 'desc_has_priv_esc',
    'desc_has_sqli', 'desc_has_xss', 'desc_has_dos',
    'desc_has_buffer_overflow', 'desc_has_path_traversal',
    'desc_has_csrf', 'desc_has_xxe'
]

vendor_features = [
    'vendor_is_high_risk', 'vendor_is_healthcare', 'vendor_risk_score'
]

interaction_features = [
    'ultimate_risk', 'critical_exploitable', 'network_accessible',
    'auth_not_required', 'high_impact_network', 'healthcare_critical'
]

all_features = basic_features + cvss_features + cwe_features + nlp_features + vendor_features + interaction_features
feature_cols = all_features

print_header('FEATURE CONFIGURATION')
print(f"Total features: {len(feature_cols)} (16 basic + 37 enhanced)")
print(f"  Basic enrichments: {len(basic_features)}")
print(f"  CVSS decomposition: {len(cvss_features)}")
print(f"  CWE intelligence: {len(cwe_features)}")
print(f"  Description NLP: {len(nlp_features)}")
print(f"  Vendor features: {len(vendor_features)}")
print(f"  Interaction features: {len(interaction_features)}")
print_separator()

# Prepare training data (features are already in DataFrames, no need to extract)
y_train = train_df['soft_label']
y_val = val_df['soft_label']
y_test = test_df['soft_label']

print(f"[OK] Training data prepared")
print(f"  Train: {len(train_df):,} CVEs")
print(f"  Val: {len(val_df):,} CVEs")
print(f"  Test: {len(test_df):,} CVEs")


FEATURE CONFIGURATION

Total features: 47 (16 basic + 37 enhanced)
  Basic enrichments: 10
  CVSS decomposition: 10
  CWE intelligence: 8
  Description NLP: 10
  Vendor features: 3
  Interaction features: 6

[OK] Training data prepared
  Train: 147,102 CVEs
  Val: 31,522 CVEs
  Test: 31,523 CVEs


## 5. Reference Scoring Models

Simple reference scorers for all-model comparison.



In [6]:
# Reference scorer 1: CVSS-only ranker
print('Computing CVSS-only reference scores...')
cvss_scores_val = compute_cvss_only_scores(val_df)
cvss_scores_test = compute_cvss_only_scores(test_df)

# Reference scorer 2: Heuristic ranker (weighted combination of signals)
print('Computing heuristic reference scores...')
heuristic_scores_val = compute_heuristic_scores(val_df)
heuristic_scores_test = compute_heuristic_scores(test_df)

print()
print('[OK] Reference scores computed')
print('  CVSS reference: Simple CVSS score ranking')
print('  Heuristic reference: 0.35*CVSS + 0.30*EPSS + 0.20*KEV + 0.10*recency + 0.05*ATT&CK')

print_header('REFERENCE SCORE DISTRIBUTIONS (Validation Set)')
print('CVSS-only scores:')
print(f"  Min: {cvss_scores_val.min():.3f}")
print(f"  Max: {cvss_scores_val.max():.3f}")
print(f"  Mean: {cvss_scores_val.mean():.3f}")
print(f"  Median: {np.median(cvss_scores_val):.3f}")

print()
print('Heuristic scores:')
print(f"  Min: {heuristic_scores_val.min():.3f}")
print(f"  Max: {heuristic_scores_val.max():.3f}")
print(f"  Mean: {heuristic_scores_val.mean():.3f}")
print(f"  Median: {np.median(heuristic_scores_val):.3f}")


trace_stage_done(
    'reference_models',
    status='ok',
    val_rows=int(len(val_df)),
    test_rows=int(len(test_df)),
    cvss_val_mean=float(np.mean(cvss_scores_val)),
    heuristic_val_mean=float(np.mean(heuristic_scores_val))
)

Computing CVSS-only reference scores...
Computing heuristic reference scores...

[OK] Reference scores computed
  CVSS reference: Simple CVSS score ranking
  Heuristic reference: 0.35*CVSS + 0.30*EPSS + 0.20*KEV + 0.10*recency + 0.05*ATT&CK

REFERENCE SCORE DISTRIBUTIONS (Validation Set)

CVSS-only scores:
  Min: 0.000
  Max: 1.000
  Mean: 0.664
  Median: 0.650

Heuristic scores:
  Min: 0.085
  Max: 0.966
  Mean: 0.344
  Median: 0.345
[TRACE] reference_models | ok | {'val_rows': 31522, 'test_rows': 31523, 'cvss_val_mean': 0.6637386587145486, 'heuristic_val_mean': 0.3435429265725443}


## 6. XGBoost Ranker

Gradient-boosted trees with `rank:ndcg` objective - direct NDCG ranking optimisation, trained independently of LambdaMART to provide an uncorrelated comparison reference.



In [7]:
import xgboost as xgb

# Ensure grouping key exists even if LambdaMART cell has not run yet.
for _df in (train_df, val_df, test_df):
    if 'published_week' not in _df.columns:
        _df['published_week'] = _df['published'].dt.tz_localize(None).dt.to_period('W').astype(str)

# Ensure matrices exist for this cell to run independently.
X_train_array = train_df[feature_cols].fillna(0).values
X_val_array = val_df[feature_cols].fillna(0).values
X_test_array = test_df[feature_cols].fillna(0).values

# Train XGBoost Ranker (same objective: rank:ndcg)
print("\nTraining XGBoost ranker (objective=rank:ndcg)...")
xgb_params = {
    'objective': 'rank:ndcg',
    'eval_metric': 'ndcg',
    'eta': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}
train_group = train_df.groupby('published_week').size().values.tolist()
val_group = val_df.groupby('published_week').size().values.tolist()
dtrain = xgb.DMatrix(X_train_array, label=y_train.values, feature_names=feature_cols)
dval = xgb.DMatrix(X_val_array, label=y_val.values, feature_names=feature_cols)
dtest = xgb.DMatrix(X_test_array, label=y_test.values, feature_names=feature_cols)
dtrain.set_group(train_group)
dval.set_group(val_group)
xgb_model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=500,
    evals=[(dval, 'valid')],
    early_stopping_rounds=30,
    verbose_eval=False
)
xgb_scores_val = xgb_model.predict(dval)
xgb_scores_test = xgb_model.predict(dtest)
print(f"[OK] XGBoost training complete (best_iteration={xgb_model.best_iteration})")

trace_stage_done(
    'xgboost_ranker_train',
    status='ok',
    best_iteration=int(xgb_model.best_iteration),
    train_groups=int(len(train_group)),
    val_groups=int(len(val_group))
)


Training XGBoost ranker (objective=rank:ndcg)...
[OK] XGBoost training complete (best_iteration=65)
[TRACE] xgboost_ranker_train | ok | {'best_iteration': 65, 'train_groups': 341, 'val_groups': 40}


In [8]:
# Visualize XGBoost training curve
if 'xgb_model' in globals() and hasattr(xgb_model, 'evals_result'):
    xgb_hist = xgb_model.evals_result()
    val_key    = 'valid' if 'valid' in xgb_hist else next(iter(xgb_hist.keys()))
    metric_key = 'ndcg'  if 'ndcg'  in xgb_hist[val_key] else next(iter(xgb_hist[val_key].keys()))
    val_metric = xgb_hist[val_key][metric_key]

    iterations = list(range(1, len(val_metric) + 1))
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=iterations, y=val_metric,
        mode='lines', name=f'Validation {metric_key.upper()}',
        line=dict(color='#3498DB', width=2)
    ))

    best_iter  = int(getattr(xgb_model, 'best_iteration', len(val_metric) - 1)) + 1
    best_idx   = max(0, min(best_iter - 1, len(val_metric) - 1))
    best_score = val_metric[best_idx]
    fig.add_trace(go.Scatter(
        x=[best_iter], y=[best_score],
        mode='markers', name=f'Best (iter {best_iter})',
        marker=dict(color='red', size=10, symbol='star')
    ))

    fig.update_layout(
        title='XGBoost Training Curve',
        xaxis_title='Iteration',
        yaxis_title=metric_key.upper(),
        height=400,
        hovermode='x unified'
    )

    save_plot(fig, 'xgb_training_curve')
    print('[OK] XGBoost training curve saved')
else:
    print('[WARN] XGBoost model not found — run Section 6 first.')


[WARN] XGBoost model not found — run Section 6 first.


## 7. Graph Construction (Training Scope)

Build the CVE relationship graphs that power DiffusionRank and RGCN.

**Data scope**: graphs are built on the *training split only* so graph topology cannot leak test-set label information.

Graph types:
1. **CVE–CWE Bipartite Graph**: connects CVEs to their weakness types (CWE IDs)
2. **CVE Similarity Graph**: connects CVEs with high cosine-similarity feature vectors

In [9]:
import numpy as np
import pandas as pd
import torch
import networkx as nx
from scipy.sparse import coo_matrix
from sklearn.neighbors import NearestNeighbors

trace_event('graph_build', status='start')
print_header('HIGH-PERFORMANCE GRAPH CONSTRUCTION (TRAINING SCOPE)')

# 1) Vectorized CVE-CWE relations
print(f"[PROCESS] Vectorizing CVE-CWE relations for {len(train_df):,} rows...")

bipartite_df = train_df[['cve_id', 'cwe']].copy()
bipartite_df['cve_id'] = bipartite_df['cve_id'].astype(str)
bipartite_df['cwe'] = bipartite_df['cwe'].astype(str).str.split(',')
relations = bipartite_df.explode('cwe', ignore_index=True)
relations['cwe'] = relations['cwe'].astype(str).str.strip()
relations = relations[~relations['cwe'].isin(['nan', 'None', ''])].copy()

cve_ids = train_df['cve_id'].astype(str).drop_duplicates().to_numpy()
cve_map = {val: i for i, val in enumerate(cve_ids)}
unique_cwes = relations['cwe'].drop_duplicates().to_numpy()
cwe_map = {val: i + len(cve_map) for i, val in enumerate(unique_cwes)}

src_indices = relations['cve_id'].map(cve_map).to_numpy(dtype=np.int64, copy=False)
dst_indices = relations['cwe'].map(cwe_map).to_numpy(dtype=np.int64, copy=False)

G_bipartite = nx.Graph()
G_bipartite.add_nodes_from((cid, {'node_type': 'cve'}) for cid in cve_map.keys())
G_bipartite.add_nodes_from((cwe, {'node_type': 'cwe'}) for cwe in cwe_map.keys())
G_bipartite.add_edges_from(zip(relations['cve_id'].tolist(), relations['cwe'].tolist()))

cve_nodes_set = set(cve_map.keys())
cwe_nodes_set = set(cwe_map.keys())

print(f"[OK] Bipartite structure created:")
print(f"  CVE Nodes: {len(cve_map):,}")
print(f"  CWE Nodes: {len(cwe_map):,}")
print(f"  Edges    : {G_bipartite.number_of_edges():,}")

# 2) Chunked kNN similarity over full training set
GRAPH_TOP_K = int(exp_cfg.similarity.k_neighbors)
GRAPH_THRESHOLD = float(exp_cfg.similarity.threshold)
sim_cfg = getattr(exp_cfg, 'similarity', None)
if hasattr(sim_cfg, 'get'):
    GRAPH_KNN_CHUNK_SIZE = int(sim_cfg.get('chunk_size', 2000))
else:
    GRAPH_KNN_CHUNK_SIZE = int(getattr(sim_cfg, 'chunk_size', 2000)) if sim_cfg is not None else 2000

print(f"\n[PROCESS] Building Similarity Graph (k={GRAPH_TOP_K}, chunk={GRAPH_KNN_CHUNK_SIZE})...")

exclude_cols = {'cve_id', 'published', 'modified', 'label', 'confidence', 'label_confidence', 'soft_label', 'cvss_vector', 'cwe', 'published_week'}
feat_cols = [c for c in train_df.columns if c not in exclude_cols and np.issubdtype(train_df[c].dtype, np.number)]
X = train_df[feat_cols].fillna(0).to_numpy(dtype=np.float32, copy=False)

nn = NearestNeighbors(n_neighbors=GRAPH_TOP_K + 1, metric='cosine', algorithm='brute', n_jobs=-1)
nn.fit(X)

sim_src_parts = []
sim_dst_parts = []
sim_weight_parts = []

for start in range(0, X.shape[0], GRAPH_KNN_CHUNK_SIZE):
    end = min(start + GRAPH_KNN_CHUNK_SIZE, X.shape[0])
    distances, indices = nn.kneighbors(X[start:end], return_distance=True)
    neighbor_idx = indices[:, 1:]
    sims = 1.0 - distances[:, 1:]
    rows, cols = np.where(sims >= GRAPH_THRESHOLD)

    if rows.size == 0:
        continue

    src_chunk = rows.astype(np.int64, copy=False) + start
    dst_chunk = neighbor_idx[rows, cols].astype(np.int64, copy=False)
    w_chunk = sims[rows, cols].astype(np.float32, copy=False)

    not_self = src_chunk != dst_chunk
    if np.any(not_self):
        sim_src_parts.append(src_chunk[not_self])
        sim_dst_parts.append(dst_chunk[not_self])
        sim_weight_parts.append(w_chunk[not_self])

if sim_src_parts:
    sim_src = np.concatenate(sim_src_parts)
    sim_dst = np.concatenate(sim_dst_parts)
    sim_weights = np.concatenate(sim_weight_parts)
else:
    sim_src = np.array([], dtype=np.int64)
    sim_dst = np.array([], dtype=np.int64)
    sim_weights = np.array([], dtype=np.float32)

print(f"[OK] Similarity edges generated: {len(sim_src):,}")

sim_adj = coo_matrix((sim_weights, (sim_src, sim_dst)), shape=(len(cve_ids), len(cve_ids)))
G_similarity_idx = nx.from_scipy_sparse_array(sim_adj, create_using=nx.Graph())
idx_to_cve = {i: cid for i, cid in enumerate(cve_ids)}
G_similarity = nx.relabel_nodes(G_similarity_idx, idx_to_cve)
G_similarity.add_nodes_from(cve_ids.tolist())

print(f"[OK] CVE Similarity Graph:")
print(f"  Nodes : {G_similarity.number_of_nodes():,}")
print(f"  Edges : {G_similarity.number_of_edges():,}")

# 3) CWE projection for diffusion fallback
MAX_CWE_GROUP = 100
MAX_PROJ_EDGES = 2_000_000
cwe_groups = relations.groupby('cwe')['cve_id'].agg(list)

G_projected = nx.Graph()
G_projected.add_nodes_from(cve_nodes_set)
proj_edges = 0

for cve_group in cwe_groups:
    if len(cve_group) > MAX_CWE_GROUP:
        continue
    for ci in range(len(cve_group)):
        for cj in range(ci + 1, len(cve_group)):
            G_projected.add_edge(cve_group[ci], cve_group[cj])
            proj_edges += 1
            if proj_edges >= MAX_PROJ_EDGES:
                break
        if proj_edges >= MAX_PROJ_EDGES:
            break
    if proj_edges >= MAX_PROJ_EDGES:
        break

graph_for_diffusion = G_projected if G_projected.number_of_edges() > 0 else G_similarity
print(f"[OK] Diffusion graph (bounded CWE projection): {graph_for_diffusion.number_of_edges():,} edges")

# 4) Optional tensor consolidation for graph-model interop
edge_src = np.concatenate([src_indices, sim_src]).astype(np.int64, copy=False)
edge_dst = np.concatenate([dst_indices, sim_dst]).astype(np.int64, copy=False)
edge_type_np = np.concatenate([
    np.zeros(len(src_indices), dtype=np.int64),
    np.ones(len(sim_src), dtype=np.int64)
])

edge_index = torch.tensor(np.stack([edge_src, edge_dst]), dtype=torch.long)
edge_type = torch.tensor(edge_type_np, dtype=torch.long)

trace_event(
    'graph_build',
    status='ok',
    total_edges=int(edge_index.shape[1]),
    bipartite_edges=int(G_bipartite.number_of_edges()),
    sim_edges=int(G_similarity.number_of_edges()),
    proj_edges=int(graph_for_diffusion.number_of_edges()),
    cve_nodes=int(len(cve_map)),
    cwe_nodes=int(len(cwe_map))
)

print(f"\n[FINAL] Ready for downstream graph models.")
print(f"  edge_index shape: {tuple(edge_index.shape)}")

[TRACE] graph_build | start | 

HIGH-PERFORMANCE GRAPH CONSTRUCTION (TRAINING SCOPE)

[PROCESS] Vectorizing CVE-CWE relations for 147,102 rows...
[OK] Bipartite structure created:
  CVE Nodes: 147,102
  CWE Nodes: 652
  Edges    : 145,337

[PROCESS] Building Similarity Graph (k=10, chunk=2000)...
[OK] Similarity edges generated: 1,441,633
[OK] CVE Similarity Graph:
  Nodes : 147,102
  Edges : 991,988
[OK] Diffusion graph (bounded CWE projection): 130,431 edges
[TRACE] graph_build | ok | {'total_edges': 1586970, 'bipartite_edges': 145337, 'sim_edges': 991988, 'proj_edges': 130431, 'cve_nodes': 147102, 'cwe_nodes': 652}

[FINAL] Ready for downstream graph models.
  edge_index shape: (2, 1586970)


## 8. DiffusionRank: Graph-Based Priority Propagation

Random-walk-with-restart propagates LambdaMART scores through the CVE–CWE graph. All seed scores come from the already-trained LambdaMART model — DiffusionRank only re-ranks; it does not learn new parameters.

**Output**: `diffusion_scores_test` — re-ranked scores for the test set.

In [10]:
from src.models.diffusion_rank import diffusion_rank

trace_event('diffusion_rank', status='start')
print_header('DIFFUSIONRANK (GRAPH-AUGMENTED RE-RANKING)')

try:
    # Seed scores always come from XGBoost (Section 6 — always trained before Section 8).
    # LambdaMART is trained later in Section 11; do NOT depend on it here.
    if 'xgb_model' not in globals() or 'xgb_scores_test' not in globals():
        raise RuntimeError(
            'XGBoost model/scores not found. Run Section 6 (XGBoost Ranker) first.'
        )

    import xgboost as _xgb_mod

    df_all   = pd.concat([train_df, val_df, test_df], ignore_index=True)
    X_all    = df_all[feature_cols].fillna(0).values
    d_all    = _xgb_mod.DMatrix(X_all, feature_names=feature_cols)
    all_base_scores  = xgb_model.predict(d_all)
    test_base_scores = np.asarray(xgb_scores_test)
    base_model       = 'XGBoost'

    seed_scores_all = dict(zip(df_all['cve_id'], all_base_scores))
    test_base_map   = dict(zip(test_df['cve_id'], test_base_scores))

    # Restrict seed_scores to nodes present in the diffusion graph (train CVEs)
    seed_scores = {k: v for k, v in seed_scores_all.items() if k in cve_nodes_set}
    if not seed_scores:
        raise RuntimeError('No overlap between seed scores and diffusion graph nodes.')

    print(f"  Seed nodes : {len(seed_scores):,}")
    print(f"  Graph nodes: {graph_for_diffusion.number_of_nodes():,}")
    print(f"  Base ranker: {base_model}")
    print(f"  Alpha (restart prob): 0.85")

    diffusion_scores_map = diffusion_rank(
        graph_for_diffusion, seed_scores, alpha=0.85, max_iter=100, tol=1e-6
    )

    # Map back onto test_df (nodes not in train graph get XGBoost fallback score)
    diffusion_scores_test = np.array([
        diffusion_scores_map.get(cid, test_base_map.get(cid, 0.0))
        for cid in test_df['cve_id']
    ])
    train_df['diffusion_score'] = [diffusion_scores_map.get(cid, 0.0) for cid in train_df['cve_id']]
    test_df['diffusion_score']  = diffusion_scores_test

    corr = float(np.corrcoef(test_base_scores, diffusion_scores_test)[0, 1])
    print(f"\n[OK] DiffusionRank complete")
    print(f"  Test score range: [{diffusion_scores_test.min():.6f}, {diffusion_scores_test.max():.6f}]")
    print(f"  Correlation vs {base_model} test scores: {corr:.4f}")
    trace_event('diffusion_rank', status='ok', corr=round(corr, 4))

except Exception as e:
    print(f"[WARN] DiffusionRank failed: {e}")
    # Fallback: use XGBoost scores directly (available from Section 6)
    if 'xgb_scores_test' in globals():
        diffusion_scores_test = np.asarray(xgb_scores_test).copy()
        fallback_used = 'xgb_scores_test'
    elif 'test_df' in globals():
        diffusion_scores_test = np.zeros(len(test_df), dtype=float)
        fallback_used = 'zeros'
    else:
        diffusion_scores_test = np.array([], dtype=float)
        fallback_used = 'empty'

    if 'test_df' in globals() and len(diffusion_scores_test) == len(test_df):
        test_df['diffusion_score'] = diffusion_scores_test

    print(f"[WARN] Using fallback diffusion scores: {fallback_used}")
    trace_event('diffusion_rank', status='error', error=str(e), fallback=fallback_used)

[TRACE] diffusion_rank | start | 

DIFFUSIONRANK (GRAPH-AUGMENTED RE-RANKING)

  Seed nodes : 147,102
  Graph nodes: 147,102
  Base ranker: XGBoost
  Alpha (restart prob): 0.85

[OK] DiffusionRank complete
  Test score range: [-2.869931, 2.749571]
  Correlation vs XGBoost test scores: 1.0000
[TRACE] diffusion_rank | ok | {'corr': 1.0}


## 9. RGCN: Relational Graph Convolutional Network

Two-layer graph neural network that learns CVE embeddings from the CWE relational structure. Trained on the training split; evaluated on the test split.

**Output**: `rgcn_scores_test` — node classification probability scores for the test set.

In [11]:
import gc
import os
from array import array
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

trace_fn = trace_event
trace_fn('rgcn_train', status='start')

print_header('RGCN: MEMORY-EFFICIENT FULL-GRAPH TRAINING')


def build_rgcn_bipartite_edges(source_df: pd.DataFrame, cve_id_to_idx: dict, max_cwes_per_cve: int = 20):
    """Build CVE<->CWE bipartite edges in O(N) associations using compact buffers."""
    cwe_to_idx = {}
    src_buf = array('I')
    dst_buf = array('I')

    for _, row in source_df[['cve_id', 'cwe']].iterrows():
        cve_id = str(row['cve_id'])
        if cve_id not in cve_id_to_idx:
            continue

        cwe_value = row.get('cwe')
        if pd.isna(cwe_value):
            continue

        if isinstance(cwe_value, str):
            cwe_tokens = [t.strip() for t in cwe_value.split(',') if t.strip() and t.strip() != 'nan']
        else:
            tok = str(cwe_value).strip()
            cwe_tokens = [tok] if tok and tok != 'nan' else []

        if not cwe_tokens:
            continue

        cwe_tokens = cwe_tokens[:max_cwes_per_cve]
        cve_idx = cve_id_to_idx[cve_id]

        for cwe in cwe_tokens:
            if cwe not in cwe_to_idx:
                cwe_to_idx[cwe] = len(cwe_to_idx)
            cwe_idx = cwe_to_idx[cwe]
            src_buf.append(cve_idx)
            dst_buf.append(cwe_idx)

    if len(src_buf) == 0:
        raise ValueError('No CVE-CWE relations found for bipartite graph.')

    # Copy into persistent NumPy arrays (safer than view-based frombuffer).
    src_np = np.array(src_buf, dtype=np.int64)
    dst_np = np.array(dst_buf, dtype=np.int64)
    return src_np, dst_np, cwe_to_idx


def build_node_features(
    source_df: pd.DataFrame,
    numeric_cols: list,
    total_nodes: int,
    n_cve_nodes: int,
    feature_tmp_dir: Path,
    memmap_threshold_bytes: int = 1_500_000_000,
    chunk_size: int = 200_000,
):
    """Create standardized feature matrix with optional memmap backing when estimated size is large."""
    feat_dim = len(numeric_cols)
    est_bytes = int(total_nodes * feat_dim * 4)

    feature_tmp_dir.mkdir(parents=True, exist_ok=True)
    use_memmap = est_bytes >= memmap_threshold_bytes

    scaler = StandardScaler()
    fit_matrix = source_df[numeric_cols].fillna(0).to_numpy(dtype=np.float32, copy=False)
    scaler.fit(fit_matrix)

    if use_memmap:
        mmap_path = feature_tmp_dir / 'rgcn_node_features.float32.mmap'
        x_np = np.memmap(mmap_path, mode='w+', dtype=np.float32, shape=(total_nodes, feat_dim))
        x_np[:] = 0.0

        for start in range(0, n_cve_nodes, chunk_size):
            end = min(start + chunk_size, n_cve_nodes)
            chunk = source_df.iloc[start:end][numeric_cols].fillna(0).to_numpy(dtype=np.float32, copy=False)
            x_np[start:end] = scaler.transform(chunk).astype(np.float32, copy=False)
        x_np.flush()
        return x_np, True

    x_np = np.zeros((total_nodes, feat_dim), dtype=np.float32)
    x_np[:n_cve_nodes] = scaler.transform(fit_matrix).astype(np.float32, copy=False)
    return x_np, False


def train_rgcn_large_scale(
    source_df: pd.DataFrame,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    trace_cb,
    feature_tmp_dir: Path,
    batch_size: int = 1024,
    num_neighbors: list = [15, 10],
    hidden_channels: int = 64,
    learning_rate: float = 0.005,
    max_epochs: int = 20,
    patience: int = 5,
):
    try:
        from torch_geometric.nn import RGCNConv
        from torch_geometric.utils import coalesce
    except Exception as e:
        raise RuntimeError(f"torch_geometric is required for RGCN training: {e}")

    # Guardrail: ensure source_df covers all CVEs used by train/val/test splits.
    required_ids = (
        set(train_df['cve_id'].astype(str).tolist())
        | set(val_df['cve_id'].astype(str).tolist())
        | set(test_df['cve_id'].astype(str).tolist())
    )
    source_df_local = source_df.copy()
    source_df_local['cve_id'] = source_df_local['cve_id'].astype(str)
    available_ids = set(source_df_local['cve_id'].tolist())
    missing_ids = required_ids - available_ids
    if missing_ids:
        print(f"[WARN] source_df missing {len(missing_ids):,} CVE IDs required by splits; attempting auto-expand from global df")
        if 'df' in globals():
            global_df = globals()['df'].copy()
            global_df['cve_id'] = global_df['cve_id'].astype(str)
            add_rows = global_df[global_df['cve_id'].isin(missing_ids)].copy()
            if not add_rows.empty:
                source_df_local = pd.concat([source_df_local, add_rows], ignore_index=True)
                source_df_local = source_df_local.drop_duplicates(subset='cve_id', keep='last').copy()
                available_ids = set(source_df_local['cve_id'].tolist())
                missing_ids = required_ids - available_ids
                print(f"[INFO] Auto-expanded source_df by {len(add_rows):,} rows")
        if missing_ids:
            print(f"[WARN] Still missing {len(missing_ids):,} CVE IDs after auto-expand")

    exclude_cols = {
        'cve_id', 'published', 'modified', 'label', 'confidence', 'label_confidence',
        'soft_label', 'cvss_vector', 'cwe', 'published_week', 'ltr_score',
        'diffusion_score', 'rgcn_score', 'bootstrap_mean', 'bootstrap_std',
        'ensemble_simple', 'ensemble_weighted'
    }
    numeric_cols = [
        col for col in source_df_local.columns
        if col not in exclude_cols and pd.api.types.is_numeric_dtype(source_df_local[col])
    ]
    if not numeric_cols:
        raise ValueError('No numeric feature columns available for RGCN.')

    source_df_local = source_df_local.drop_duplicates(subset='cve_id', keep='last').copy()
    source_df_local['cve_id'] = source_df_local['cve_id'].astype(str)

    cve_ids = source_df_local['cve_id'].tolist()
    cve_id_to_idx = {cid: i for i, cid in enumerate(cve_ids)}
    n_cve_nodes = len(cve_ids)

    src_cve, dst_cwe_raw, cwe_to_idx = build_rgcn_bipartite_edges(source_df_local, cve_id_to_idx)
    n_cwe_nodes = len(cwe_to_idx)
    total_nodes = n_cve_nodes + n_cwe_nodes

    dst_cwe = dst_cwe_raw + n_cve_nodes

    edge_src = np.concatenate([src_cve, dst_cwe]).astype(np.int64, copy=False)
    edge_dst = np.concatenate([dst_cwe, src_cve]).astype(np.int64, copy=False)
    edge_type_np = np.concatenate([
        np.zeros(len(src_cve), dtype=np.int64),
        np.ones(len(src_cve), dtype=np.int64),
    ])

    edge_index = torch.tensor(np.vstack([edge_src, edge_dst]), dtype=torch.long)
    edge_type = torch.tensor(edge_type_np, dtype=torch.long)
    edge_index, edge_type = coalesce(edge_index, edge_type, num_nodes=total_nodes)

    x_np, used_memmap = build_node_features(
        source_df_local,
        numeric_cols,
        total_nodes,
        n_cve_nodes,
        feature_tmp_dir=feature_tmp_dir,
    )

    y_np = np.full(total_nodes, -1.0, dtype=np.float32)
    y_np[:n_cve_nodes] = source_df_local['soft_label'].fillna(0).astype(float).to_numpy(dtype=np.float32, copy=False)

    train_idx = np.array([
        cve_id_to_idx[cid]
        for cid in train_df['cve_id'].astype(str).tolist()
        if cid in cve_id_to_idx
    ], dtype=np.int64)
    val_idx = np.array([
        cve_id_to_idx[cid]
        for cid in val_df['cve_id'].astype(str).tolist()
        if cid in cve_id_to_idx
    ], dtype=np.int64)
    test_idx = np.array([
        cve_id_to_idx[cid]
        for cid in test_df['cve_id'].astype(str).tolist()
        if cid in cve_id_to_idx
    ], dtype=np.int64)

    if len(train_idx) < 100:
        raise ValueError(f'Insufficient training nodes for RGCN: {len(train_idx)}')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    x_tensor = torch.from_numpy(np.asarray(x_np, dtype=np.float32)).to(device)
    y_tensor = torch.from_numpy(y_np).to(device)
    edge_index = edge_index.to(device)
    edge_type = edge_type.to(device)

    train_idx_t = torch.from_numpy(train_idx).long().to(device)
    val_idx_t = torch.from_numpy(val_idx).long().to(device) if len(val_idx) > 0 else None
    test_idx_t = torch.from_numpy(test_idx).long().to(device)

    class FullGraphRGCN(torch.nn.Module):
        def __init__(self, in_channels: int, hidden_channels: int, num_relations: int = 2):
            super().__init__()
            self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations=num_relations)
            self.conv2 = RGCNConv(hidden_channels, 1, num_relations=num_relations)

        def forward(self, x, edge_index, edge_type):
            x = self.conv1(x, edge_index, edge_type)
            x = torch.relu(x)
            x = self.conv2(x, edge_index, edge_type)
            return x.view(-1)

    model = FullGraphRGCN(in_channels=len(numeric_cols), hidden_channels=hidden_channels).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = torch.nn.MSELoss()

    def evaluate_fullgraph():
        model.eval()
        with torch.no_grad():
            if val_idx_t is None or val_idx_t.numel() == 0:
                return float('inf')
            out = model(x_tensor, edge_index, edge_type)
            val_pred = out.index_select(0, val_idx_t)
            val_target = y_tensor.index_select(0, val_idx_t)
            valid_mask = val_target >= 0
            if not bool(valid_mask.any()):
                return float('inf')
            return float(loss_fn(val_pred[valid_mask], val_target[valid_mask]).item())

    best_val = float('inf')
    best_state = None
    stale_epochs = 0

    print(f"[INFO] Graph nodes (CVE/CWE): {n_cve_nodes:,}/{n_cwe_nodes:,}")
    print(f"[INFO] Graph edges (directed COO): {edge_index.shape[1]:,}")
    print(f"[INFO] Features: {len(numeric_cols)}")
    print(f"[INFO] Batch size parameter: {batch_size} (unused in full-graph mode)")
    print(f"[INFO] num_neighbors parameter: {num_neighbors} (unused in full-graph mode)")
    print(f"[INFO] Feature storage: {'memmap' if used_memmap else 'in-memory'}")
    print('[INFO] Sampler mode: full-graph (no NeighborLoader dependency)')

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(x_tensor, edge_index, edge_type)
        train_pred = out.index_select(0, train_idx_t)
        train_target = y_tensor.index_select(0, train_idx_t)
        train_valid = train_target >= 0
        if not bool(train_valid.any()):
            raise RuntimeError('No valid training labels available for RGCN.')

        loss = loss_fn(train_pred[train_valid], train_target[train_valid])
        loss.backward()
        optimizer.step()

        train_loss = float(loss.item())
        val_loss = evaluate_fullgraph() if len(val_idx) > 0 else train_loss
        print(f"[EPOCH {epoch:02d}] train_loss={train_loss:.6f} val_loss={val_loss:.6f}")

        if val_loss + 1e-6 < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                print(f"[INFO] Early stopping triggered at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    pred_map = {}
    model.eval()
    with torch.no_grad():
        out = model(x_tensor, edge_index, edge_type)
        test_scores_raw = out.index_select(0, test_idx_t).detach().cpu().numpy().astype(float)
        test_nodes_raw = test_idx_t.detach().cpu().numpy().astype(np.int64)
        for nid, score in zip(test_nodes_raw, test_scores_raw):
            pred_map[int(nid)] = float(score)

    pred_values = np.array([pred_map[idx] for idx in sorted(pred_map.keys())], dtype=float) if pred_map else np.array([], dtype=float)
    if pred_values.size > 0:
        pmin, pmax = float(np.nanmin(pred_values)), float(np.nanmax(pred_values))
        if pmax > pmin:
            norm_map = {k: (v - pmin) / (pmax - pmin) for k, v in pred_map.items()}
        else:
            norm_map = {k: 0.0 for k in pred_map}
    else:
        norm_map = {}

    fallback_score = float(np.nanmedian(list(norm_map.values()))) if norm_map else 0.0
    test_scores = np.array([
        norm_map.get(cve_id_to_idx.get(cid, -1), fallback_score)
        for cid in test_df['cve_id'].astype(str).tolist()
    ], dtype=float)

    trace_cb(
        'rgcn_train',
        status='ok',
        relation_source='cve_cwe_bipartite',
        train_nodes=int(len(train_idx)),
        val_nodes=int(len(val_idx)),
        test_nodes=int(len(test_idx)),
        num_edges=int(edge_index.shape[1]),
        num_features=int(len(numeric_cols)),
        used_memmap=bool(used_memmap),
    )

    del x_tensor, y_tensor, edge_index, edge_type, train_idx_t, test_idx_t
    if val_idx_t is not None:
        del val_idx_t
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

    return model, test_scores


try:
    if 'df' not in globals() or 'train_df' not in globals() or 'val_df' not in globals() or 'test_df' not in globals():
        raise ValueError('Run data loading and temporal split cells before the RGCN cell.')

    rgcn_source_df = df.copy()

    rgcn_model, rgcn_scores_test = train_rgcn_large_scale(
        source_df=rgcn_source_df,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        trace_cb=trace_fn,
        feature_tmp_dir=project_root / 'tmp',
        batch_size=512,
        num_neighbors=[15, 10],
        hidden_channels=64,
        learning_rate=0.003,
        max_epochs=15,
        patience=4,
    )

    test_df['rgcn_score'] = rgcn_scores_test

    print(f"[OK] RGCN training complete")
    print(f"[OK] RGCN test scores generated for {len(rgcn_scores_test):,} CVEs")
    print(f"[OK] RGCN score range: [{rgcn_scores_test.min():.6f}, {rgcn_scores_test.max():.6f}]")

except Exception as e:
    print(f"[WARN] RGCN failed, using zero fallback: {e}")
    if 'test_df' in globals():
        rgcn_scores_test = np.zeros(len(test_df), dtype=float)
        test_df['rgcn_score'] = rgcn_scores_test
    trace_fn('rgcn_train', status='error', error=str(e))

[TRACE] rgcn_train | start | 

RGCN: MEMORY-EFFICIENT FULL-GRAPH TRAINING

[INFO] Graph nodes (CVE/CWE): 210,147/719
[INFO] Graph edges (directed COO): 416,092
[INFO] Features: 51
[INFO] Batch size parameter: 512 (unused in full-graph mode)
[INFO] num_neighbors parameter: [15, 10] (unused in full-graph mode)
[INFO] Feature storage: in-memory
[INFO] Sampler mode: full-graph (no NeighborLoader dependency)
[EPOCH 01] train_loss=1.072494 val_loss=0.518605
[EPOCH 02] train_loss=0.700137 val_loss=0.357617
[EPOCH 03] train_loss=0.512134 val_loss=0.289180
[EPOCH 04] train_loss=0.424151 val_loss=0.251661
[EPOCH 05] train_loss=0.365931 val_loss=0.224833
[EPOCH 06] train_loss=0.314938 val_loss=0.209546
[EPOCH 07] train_loss=0.275031 val_loss=0.207984
[EPOCH 08] train_loss=0.252694 val_loss=0.215388
[EPOCH 09] train_loss=0.246648 val_loss=0.221614
[EPOCH 10] train_loss=0.247563 val_loss=0.219134
[EPOCH 11] train_loss=0.245838 val_loss=0.206204
[EPOCH 12] train_loss=0.236913 val_loss=0.186781
[EPOC

## 10. Ensemble Methods

Combine predictions from all trained models. Three strategies:
1. **Simple Average** — equal weight to every model
2. **Weighted Average** — higher weight to models with better validation NDCG
3. **Bootstrap** — 100 bootstrap samples to quantify prediction uncertainty

In [12]:
from src.models.ensemble import EnsembleRanker, bootstrap_ensemble

trace_event('ensemble_train', status='start')
print_header('ENSEMBLE METHODS')
print("[INFO] Ensemble uses models trained in Sections 5–9.")
print("[INFO] LambdaMART (Section 11) is intentionally excluded here;")
print("[INFO] it will appear in the Section 12 all-model comparison instead.")

# ── Collect scores from all models trained before this point (§5–§9) ──────────
ensemble_inputs = {}

if 'xgb_scores_test' in globals():
    ensemble_inputs['XGBoost'] = np.asarray(xgb_scores_test)
else:
    print('[WARN] XGBoost scores missing — run Section 6')

if 'diffusion_score' in test_df.columns:
    ensemble_inputs['DiffusionRank'] = test_df['diffusion_score'].values
else:
    print('[WARN] DiffusionRank scores missing — run Section 8')

if 'rgcn_score' in test_df.columns and test_df['rgcn_score'].abs().sum() > 0:
    ensemble_inputs['RGCN'] = test_df['rgcn_score'].values
else:
    print('[WARN] RGCN scores missing or all-zero — run Section 9')

if not ensemble_inputs:
    raise RuntimeError('No model scores available for ensemble. Run Sections 6–9 first.')

bad_lengths = [name for name, arr in ensemble_inputs.items() if len(arr) != len(test_df)]
if bad_lengths:
    raise RuntimeError(f'Ensemble input length mismatch for: {bad_lengths}')

print(f"\n  Models in ensemble: {list(ensemble_inputs.keys())}")
X_ens = np.column_stack(list(ensemble_inputs.values()))

# 10.1 Simple average ──────────────────────────────────────────────────────────
ensemble_simple_test = X_ens.mean(axis=1)
test_df['ensemble_simple'] = ensemble_simple_test
print("[OK] Simple Average ensemble")

# 10.2 Weighted average (val NDCG@10 as weight proxy) ─────────────────────────
from sklearn.metrics import ndcg_score as sk_ndcg

val_ndcg_weights = {}

if 'XGBoost' in ensemble_inputs:
    try:
        if 'xgb_scores_val' in globals():
            xgb_val_for_weight = np.asarray(xgb_scores_val)
        elif 'xgb_model' in globals() and 'dval' in globals():
            xgb_val_for_weight = np.asarray(xgb_model.predict(dval))
        else:
            xgb_val_for_weight = None

        if xgb_val_for_weight is not None:
            val_ndcg_weights['XGBoost'] = sk_ndcg([y_val.values], [xgb_val_for_weight], k=10)
        else:
            val_ndcg_weights['XGBoost'] = 1.0
    except Exception:
        val_ndcg_weights['XGBoost'] = 1.0

# DiffusionRank and RGCN: no direct val scores — use mean weight as proxy
default_weight = float(np.mean(list(val_ndcg_weights.values()))) if val_ndcg_weights else 1.0
for name in ensemble_inputs:
    if name not in val_ndcg_weights:
        val_ndcg_weights[name] = default_weight

w_arr = np.array([val_ndcg_weights[n] for n in ensemble_inputs])
w_arr = w_arr / w_arr.sum()
ensemble_weighted_test = X_ens @ w_arr
test_df['ensemble_weighted'] = ensemble_weighted_test
print(f"[OK] Weighted Average: weights={dict(zip(ensemble_inputs.keys(), w_arr.round(3)))}")

# 10.3 Bootstrap uncertainty on test set ──────────────────────────────────────
try:
    mean_scores, std_scores = bootstrap_ensemble(
        predictions=ensemble_inputs,
        labels=y_test.values,
        n_bootstrap=100,
        sample_size=0.8,
    )
    test_df['bootstrap_mean'] = mean_scores
    test_df['bootstrap_std']  = std_scores
    uncertainty_threshold = float(np.percentile(std_scores, 90))
    high_unc_count = int((std_scores > uncertainty_threshold).sum())
    print(f"[OK] Bootstrap uncertainty (100 iters):")
    print(f"  Mean std: {std_scores.mean():.4f}  |  High-uncertainty CVEs (top 10%): {high_unc_count:,}")
except Exception as e:
    print(f"[WARN] Bootstrap failed: {e}")

trace_event('ensemble_train', status='ok', n_models=len(ensemble_inputs))

[TRACE] ensemble_train | start | 

ENSEMBLE METHODS

[INFO] Ensemble uses models trained in Sections 5–9.
[INFO] LambdaMART (Section 11) is intentionally excluded here;
[INFO] it will appear in the Section 12 all-model comparison instead.

  Models in ensemble: ['XGBoost', 'DiffusionRank', 'RGCN']
[OK] Simple Average ensemble
[OK] Weighted Average: weights={'XGBoost': np.float64(0.333), 'DiffusionRank': np.float64(0.333), 'RGCN': np.float64(0.333)}
[OK] Bootstrap uncertainty (100 iters):
  Mean std: 0.0082  |  High-uncertainty CVEs (top 10%): 3,153
[TRACE] ensemble_train | ok | {'n_models': 3}


## 11. Learning-to-Rank: LightGBM LambdaRank (Primary Recommended Model)

Train the primary LightGBM LambdaRank ranker using the recommended temporal future-holdout protocol (train up to cutoff date, evaluate on post-cutoff data). This model is the primary artifact exported as `ltr_ranker.model`.

**What this section does**:
1. Builds temporal cutoff split (pre-cutoff train, post-cutoff holdout test)
2. Creates validation subset from training window only (leakage-safe)
3. Trains LambdaRank with confidence-weighted labels
4. Produces primary model object `ltr_model` for export and downstream use


In [13]:
# Primary recommended LambdaRank model (temporal future-holdout protocol)
print_header('PRIMARY LAMBDARANK TRAINING (RECOMMENDED TEMPORAL HOLDOUT)')

required_objs = ['df', 'feature_cols', 'train_lambdarank', 'get_default_ltr_params']
missing = [k for k in required_objs if k not in globals()]
if missing:
    raise RuntimeError(f"Missing required objects before primary LTR training: {missing}")

cutoff_date = THESIS_CUTOFF_DATE
df_primary_train = df[df['published'] <= cutoff_date].copy()
df_primary_test = df[df['published'] > cutoff_date].copy()

if len(df_primary_train) < 10 or len(df_primary_test) < 10:
    raise RuntimeError('Primary temporal split produced too few rows; check THESIS_CUTOFF_DATE and dataset window.')

df_primary_train['published_week'] = df_primary_train['published'].dt.tz_localize(None).dt.to_period('W').astype(str)
df_primary_test['published_week'] = df_primary_test['published'].dt.tz_localize(None).dt.to_period('W').astype(str)

df_primary_train = df_primary_train.sort_values('published').copy()
val_size = max(1, int(len(df_primary_train) * 0.15))
if val_size >= len(df_primary_train):
    val_size = max(1, len(df_primary_train) // 5)
split_idx = len(df_primary_train) - val_size
df_primary_train_fit = df_primary_train.iloc[:split_idx].copy()
df_primary_val = df_primary_train.iloc[split_idx:].copy()

cat_mapping_primary = fit_categorical_mapping(df_primary_train_fit, CATEGORICAL_COLS)
df_primary_train_fit = apply_categorical_mapping(df_primary_train_fit, cat_mapping_primary)
df_primary_val = apply_categorical_mapping(df_primary_val, cat_mapping_primary)
df_primary_test_enc = apply_categorical_mapping(df_primary_test, cat_mapping_primary)

print('[STATS] Primary model split:')
print(f"  Train-fit: {len(df_primary_train_fit):,} CVEs")
print(f"  Val:       {len(df_primary_val):,} CVEs")
print(f"  Holdout:   {len(df_primary_test_enc):,} CVEs")
print(f"  Cutoff date: {cutoff_date.date()}")

ltr_params = get_default_ltr_params()
ltr_model = train_lambdarank(
    df_primary_train_fit,
    df_primary_val,
    feature_cols,
    params=ltr_params,
    random_seed=42
)

# Keep Section 12 compatible by generating predictions on the standard comparison partitions
X_val_array = val_df[feature_cols].fillna(0).values
X_test_array = test_df[feature_cols].fillna(0).values
ltr_scores_val = ltr_model.predict(X_val_array)
ltr_scores_test = ltr_model.predict(X_test_array)

print('\n[OK] Primary LambdaRank model trained and assigned to ltr_model')
print(f"  Best iteration: {ltr_model.best_iteration}")
print(f"  Best validation NDCG@10: {ltr_model.best_score['valid']['ndcg@10']:.4f}")


PRIMARY LAMBDARANK TRAINING (RECOMMENDED TEMPORAL HOLDOUT)

[STATS] Primary model split:
  Train-fit: 140,831 CVEs
  Val:       24,852 CVEs
  Holdout:   44,464 CVEs
  Cutoff date: 2024-12-31

TRAINING CONFIDENCE-WEIGHTED LAMBDARANK

Feature diagnostics (train):
  Total configured features: 47
  Zero-variance features: 0
  Mostly-zero (<1% non-zero): 7
  [INFO] Mostly-zero: ['kev_flag', 'is_healthcare', 'is_curated', 'curated_severity', 'desc_has_xxe', 'vendor_is_healthcare', 'healthcare_critical']

Feature diagnostics (validation):
  Total configured features: 47
  Zero-variance features: 2
  Mostly-zero (<1% non-zero): 7
  [WARN] Zero-variance: ['is_curated', 'curated_severity']
  [INFO] Mostly-zero: ['kev_flag', 'is_healthcare', 'is_curated', 'curated_severity', 'cwe_is_crypto', 'desc_has_xxe', 'healthcare_critical']

Preparing training data...
  Train: 140,831 samples, 333 groups
  Confidence weights: min=0.200, mean=0.350, max=1.000

Preparing validation data...
  Val: 24,852 samp

## 12. All-Model Comparison on Test Set

Full comparison of all **8 trained models** on the held-out test partition. Scores computed with NDCG@K, Precision@K, Recall@K, and MAP.

Each model's **inference time (ms)** and **peak memory usage (MB)** are also recorded to support the resource-efficiency analysis in the thesis.

| Model | Type | Trained in |
|-------|------|------------|
| CVSS Reference | Baseline | Section 5 |
| Heuristic Reference | Baseline | Section 5 |
| XGBoost (rank:ndcg) | Learning-to-Rank | Section 6 |
| Graph: DiffusionRank | Graph-based | Section 8 |
| Graph: RGCN | Graph-based | Section 9 |
| Ensemble (Simple Avg) | Ensemble | Section 10 |
| Ensemble (Weighted Avg) | Ensemble | Section 10 |
| **LambdaMART (LTR)** | **Learning-to-Rank (Ours)** | **Section 11** |


In [14]:
import time
import tracemalloc

# ─────────────────────────────────────────────────────────────────────
# Section 12: All-Model Comparison on Test Set
# All 8 models: Baselines × 2, XGBoost, LambdaMART, DiffusionRank,
#               RGCN, Ensemble Simple, Ensemble Weighted
# Metrics: NDCG@K, Precision@K, Recall@K, MAP, Inference time, Peak mem
# ─────────────────────────────────────────────────────────────────────
print_header('MODEL COMPARISON - TEST SET (8 MODELS | RANKING METRICS + TIME + MEMORY)', width=82)

# ── Load metrics ──────────────────────────────────────────────────────────────
try:
    from src.evaluation.metrics import compute_flat_ranking_metrics as compute_ranking_metrics_inline
except ImportError:
    import importlib
    import src.evaluation.metrics as _metrics
    _metrics = importlib.reload(_metrics)
    if hasattr(_metrics, 'compute_flat_ranking_metrics'):
        compute_ranking_metrics_inline = _metrics.compute_flat_ranking_metrics
    else:
        raise

# ── Build candidate pool ──────────────────────────────────────────────────────
# Insertion ORDER = section order: baselines → XGBoost → graph → ensemble → LTR
# Each entry: name → (score_array, callable_for_timing | None)
candidate_models = {}

# Baselines (Section 5)
if 'cvss_scores_test' in globals():
    candidate_models['CVSS Reference'] = (np.asarray(cvss_scores_test), lambda: compute_cvss_only_scores(test_df))
else:
    print('[WARN] cvss_scores_test missing — run Section 5')

if 'heuristic_scores_test' in globals():
    candidate_models['Heuristic Reference'] = (np.asarray(heuristic_scores_test), lambda: compute_heuristic_scores(test_df))
else:
    print('[WARN] heuristic_scores_test missing — run Section 5')

# XGBoost (Section 6)
if 'xgb_scores_test' in globals():
    _xgb_arr = np.asarray(xgb_scores_test)
    if 'xgb_model' in globals() and 'dtest' in globals():
        candidate_models['XGBoost (rank:ndcg)'] = (_xgb_arr, lambda: xgb_model.predict(dtest))
    else:
        candidate_models['XGBoost (rank:ndcg)'] = (_xgb_arr, None)
elif 'xgb_model' in globals() and 'dtest' in globals():
    _xgb_arr = xgb_model.predict(dtest)
    candidate_models['XGBoost (rank:ndcg)'] = (_xgb_arr, lambda: xgb_model.predict(dtest))
else:
    print('[WARN] XGBoost scores missing — run Section 6')

# Graph: DiffusionRank (Section 8)
if 'diffusion_score' in test_df.columns:
    candidate_models['Graph: DiffusionRank'] = (test_df['diffusion_score'].values, None)
else:
    print('[WARN] DiffusionRank scores missing — run Section 8')

# Graph: RGCN (Section 9)
if 'rgcn_score' in test_df.columns and test_df['rgcn_score'].abs().sum() > 0:
    candidate_models['Graph: RGCN'] = (test_df['rgcn_score'].values, None)
else:
    print('[WARN] RGCN scores missing or all-zero — run Section 9')

# Ensemble (Section 10)
if 'ensemble_simple' in test_df.columns:
    candidate_models['Ensemble (Simple Avg)'] = (test_df['ensemble_simple'].values, None)
else:
    print('[WARN] Ensemble (Simple) missing — run Section 10')

if 'ensemble_weighted' in test_df.columns:
    candidate_models['Ensemble (Weighted Avg)'] = (test_df['ensemble_weighted'].values, None)
else:
    print('[WARN] Ensemble (Weighted) missing — run Section 10')

# LambdaMART — Primary Thesis Model (Section 11) — intentionally LAST
if 'ltr_scores_test' in globals():
    _ltr_arr = np.asarray(ltr_scores_test)
    if 'ltr_model' in globals():
        _X_t = test_df[feature_cols].fillna(0).values
        candidate_models['LambdaMART (LTR)'] = (_ltr_arr, lambda: ltr_model.predict(_X_t))
    else:
        candidate_models['LambdaMART (LTR)'] = (_ltr_arr, None)
elif 'ltr_model' in globals():
    _X_t = test_df[feature_cols].fillna(0).values
    _ltr_arr = ltr_model.predict(_X_t)
    candidate_models['LambdaMART (LTR)'] = (_ltr_arr, lambda: ltr_model.predict(_X_t))
else:
    print('[WARN] LambdaMART scores missing — run Section 11')

print(f'\n[INFO] Models in comparison: {list(candidate_models.keys())}')
print(f'[INFO] Length check (expected {len(test_df):,} per model):')
for nm, (arr, _) in candidate_models.items():
    ok = 'OK' if len(arr) == len(test_df) else f'MISMATCH ({len(arr)}!={len(test_df)})'
    print(f'  {nm:<28}: {ok}')
print()

# ── Per-model: metrics + inference timing + peak memory ──────────────────────
results        = {}
timing_results = {}

for model_name, (scores_arr, infer_fn) in candidate_models.items():
    # Inference timing
    t0 = time.perf_counter()
    if infer_fn is not None:
        _ = infer_fn()
    else:
        _ = scores_arr.copy()          # cheap array copy as timing baseline
    t_infer_ms = (time.perf_counter() - t0) * 1000

    # Peak memory during metric computation
    tracemalloc.start()
    metrics = compute_ranking_metrics_inline(y_test, scores_arr, k_values=EVAL_K_VALUES)
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    results[model_name]        = metrics
    timing_results[model_name] = {
        'inference_ms': round(t_infer_ms, 2),
        'peak_mem_mb':  round(peak_bytes / 1_048_576, 4),
    }

# ── Print results table ───────────────────────────────────────────────────────
print()
hdr = (f"{'Model':<28} {'NDCG@10':>8} {'NDCG@20':>8}"
       f" {'Prec@10':>8} {'Rec@10':>7} {'MAP':>8}"
       f" {'Time(ms)':>10} {'Mem(MB)':>9}")
print(hdr)
print('-' * len(hdr))

for m, met in results.items():
    t = timing_results[m]
    print(
        f"{m:<28}"
        f" {met.get('NDCG@10',   0):>8.4f}"
        f" {met.get('NDCG@20',   0):>8.4f}"
        f" {met.get('Precision@10', 0):>8.4f}"
        f" {met.get('Recall@10',  0):>7.4f}"
        f" {met.get('MAP',        0):>8.4f}"
        f" {t['inference_ms']:>10.1f}"
        f" {t['peak_mem_mb']:>9.4f}"
    )

# ── Save CSV ──────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).T
for col in ['inference_ms', 'peak_mem_mb']:
    results_df[col] = [timing_results[m][col] for m in results_df.index]

save_dataframe(results_df, 'model_comparison_test_results', subdir='evaluation')
model_comparison_path = project_root / 'outputs' / 'evaluation' / 'model_comparison_test_results.csv'
register_artifact('model_comparison_test_results', model_comparison_path)
print('\n[OK] Comparison saved -> outputs/evaluation/model_comparison_test_results.csv')

trace_stage_done(
    'model_comparison_test',
    status='ok',
    n_models=int(len(results)),
    artifact=str(model_comparison_path),
)


MODEL COMPARISON - TEST SET (8 MODELS | RANKING METRICS + TIME + MEMORY)


[INFO] Models in comparison: ['CVSS Reference', 'Heuristic Reference', 'XGBoost (rank:ndcg)', 'Graph: DiffusionRank', 'Graph: RGCN', 'Ensemble (Simple Avg)', 'Ensemble (Weighted Avg)', 'LambdaMART (LTR)']
[INFO] Length check (expected 31,523 per model):
  CVSS Reference              : OK
  Heuristic Reference         : OK
  XGBoost (rank:ndcg)         : OK
  Graph: DiffusionRank        : OK
  Graph: RGCN                 : OK
  Ensemble (Simple Avg)       : OK
  Ensemble (Weighted Avg)     : OK
  LambdaMART (LTR)            : OK


Model                         NDCG@10  NDCG@20  Prec@10  Rec@10      MAP   Time(ms)   Mem(MB)
---------------------------------------------------------------------------------------------
CVSS Reference                 0.3312   0.3501   0.0000  0.0000   0.0421        0.0    2.6872
Heuristic Reference            0.8773   0.9163   1.0000  0.0441   0.7875        0.8    3.1550
XGBoost (ran

In [15]:
# Visualize model comparison
metrics_to_plot = ['NDCG@10', 'NDCG@20', 'Precision@10', 'Precision@20', 'MAP']

comparison_data = []
for model_name, metrics in results.items():
    for metric_name in metrics_to_plot:
        if metric_name in metrics:
            comparison_data.append({
                'Model': model_name,
                'Metric': metric_name,
                'Score': metrics[metric_name]
            })

comparison_df = pd.DataFrame(comparison_data)

fig = px.bar(
    comparison_df,
    x='Metric',
    y='Score',
    color='Model',
    barmode='group',
    title='Model Comparison: Ranking Metrics',
    labels={'Score': 'Score', 'Metric': ''},
    text='Score'
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(height=450, yaxis_range=[0, 1.1])

save_plot(fig, 'model_comparison_test')
print("[OK] Model comparison plot saved")

[OK] Model comparison plot saved


## 13. Statistical Significance Testing

Validate whether performance differences between models are statistically meaningful using paired Wilcoxon signed-rank tests.

In [16]:
# Wilcoxon signed-rank test on per-group ranking quality (paired comparison)
print_header('STATISTICAL SIGNIFICANCE TESTS')
print('Wilcoxon Signed-Rank Test (per-week NDCG@20):')
print('H0: No difference in ranking quality')
print('HA: LambdaMART produces different rankings')
print()

# per_group_ndcg is in src/evaluation/metrics.py
from src.evaluation.metrics import per_group_ndcg


sig_rows = []
score_map = {
    'CVSS Reference': cvss_scores_test,
    'Heuristic Reference': heuristic_scores_test,
}
if 'ltr_scores_test' in globals():
    score_map['Learning-to-Rank: LambdaMART'] = ltr_scores_test
if 'xgb_scores_test' in globals():
    score_map['Learning-to-Rank: XGBoost'] = xgb_scores_test

sig_df_base = test_df.copy()
for name, arr in score_map.items():
    key = name.lower().replace(' ', '_').replace('-', '_').replace(':', '')
    sig_df_base[f'_score_{key}'] = arr

if '_score_learning_to_rank_lambdamart' not in sig_df_base.columns:
    print('[WARN] LambdaMART scores not available -- Wilcoxon SRT requires LambdaMART as reference; skipping')
else:
    ref_col = '_score_learning_to_rank_lambdamart'
    ref = per_group_ndcg(sig_df_base, ref_col, k=20)
    for challenger in ['CVSS Reference', 'Heuristic Reference', 'Learning-to-Rank: XGBoost']:
        key = challenger.lower().replace(' ', '_').replace('-', '_').replace(':', '')
        col = f'_score_{key}'
        if col not in sig_df_base.columns:
            continue
        ch = per_group_ndcg(sig_df_base, col, k=20)
        pair = ref.merge(ch, on='group', suffixes=('_ref', '_ch'))
        if len(pair) < 5:
            print(f"  [WARN] LambdaMART vs {challenger}: insufficient paired groups (n={len(pair)})")
            continue

        try:
            stat, p = wilcoxon(pair['ndcg_ref'], pair['ndcg_ch'], alternative='two-sided')
        except ValueError:
            stat, p = float('nan'), 1.0

        print(f"  Learning-to-Rank: LambdaMART vs {challenger}:")
        print(f"    Paired groups: {len(pair)}")
        print(f"    Statistic: {stat:.4f}")
        print(f"    p-value: {p:.6f}")
        print(f"    Result: {'[OK] Significant' if p < 0.05 else '[INFO] Not significant'} (alpha=0.05)")

        sig_rows.append({
            'Comparison': f'Learning-to-Rank: LambdaMART vs {challenger}',
            'N_groups': int(len(pair)),
            'LambdaMART_mean_ndcg20': float(pair['ndcg_ref'].mean()),
            f'{challenger}_mean_ndcg20': float(pair['ndcg_ch'].mean()),
            'Statistic': float(stat),
            'p_value': float(p),
            'Significant_alpha_0_05': bool(p < 0.05),
        })

if sig_rows:
    significance_df = pd.DataFrame(sig_rows)
    save_dataframe(significance_df, 'significance_tests_per_group_ndcg20', subdir='evaluation')
    print('\n[OK] Saved: outputs/evaluation/significance_tests_per_group_ndcg20.csv')

print()
print_separator()
print()


STATISTICAL SIGNIFICANCE TESTS

Wilcoxon Signed-Rank Test (per-week NDCG@20):
H0: No difference in ranking quality
HA: LambdaMART produces different rankings

  Learning-to-Rank: LambdaMART vs CVSS Reference:
    Paired groups: 39
    Statistic: 0.0000
    p-value: 0.000000
    Result: [OK] Significant (alpha=0.05)
  Learning-to-Rank: LambdaMART vs Heuristic Reference:
    Paired groups: 39
    Statistic: 3.0000
    p-value: 0.000001
    Result: [OK] Significant (alpha=0.05)
  Learning-to-Rank: LambdaMART vs Learning-to-Rank: XGBoost:
    Paired groups: 39
    Statistic: 17.0000
    p-value: 0.885823
    Result: [INFO] Not significant (alpha=0.05)
[OK] DataFrame saved: outputs/evaluation/significance_tests_per_group_ndcg20.csv

[OK] Saved: outputs/evaluation/significance_tests_per_group_ndcg20.csv





## 14. Top-K Analysis

Inspect highest-ranked CVEs from each model to compare practical triage quality (high-priority capture, KEV presence, and confidence).

In [17]:
# Analyze top-20 recommendations from each model
print_header('TOP-20 RECOMMENDATIONS ANALYSIS')

test_df_copy = test_df.copy()
if 'ltr_scores_test' in globals():
    test_df_copy['ltr_score'] = ltr_scores_test
if 'xgb_scores_test' in globals():
    test_df_copy['xgb_score'] = xgb_scores_test
test_df_copy['cvss_score'] = cvss_scores_test
test_df_copy['heuristic_score'] = heuristic_scores_test

score_cols = [('Heuristic', 'heuristic_score'), ('CVSS', 'cvss_score')]
if 'ltr_score' in test_df_copy.columns:
    score_cols = [('LambdaMART', 'ltr_score')] + score_cols
if 'xgb_score' in test_df_copy.columns:
    score_cols = [('XGBoost', 'xgb_score')] + score_cols

for model_name, score_col in score_cols:
    top20 = test_df_copy.nlargest(20, score_col)
    
    print(f"\n{model_name}:")
    print(f"  High-priority (label≥2): {(top20['soft_label'] >= 2).sum()}/20 ({(top20['soft_label'] >= 2).mean()*100:.1f}%)")
    print(f"  KEV flags: {top20['kev_flag'].sum()}/20 ({top20['kev_flag'].mean()*100:.1f}%)")
    print(f"  Mean CVSS: {top20['cvss'].mean():.2f}")
    print(f"  Mean confidence: {top20['label_confidence'].mean():.3f}")

print_separator()

# Save top-20 from best available LTR model
if 'ltr_score' in test_df_copy.columns:
    top20_ltr = test_df_copy.nlargest(20, 'ltr_score')[['cve_id', 'published', 'cvss', 'epss_score', 'kev_flag', 'soft_label', 'ltr_score']]
    save_dataframe(top20_ltr, 'top20_ltr_recommendations', subdir='evaluation')
    print("[OK] Top-20 LTR recommendations saved")
elif 'xgb_score' in test_df_copy.columns:
    top20_xgb = test_df_copy.nlargest(20, 'xgb_score')[['cve_id', 'published', 'cvss', 'epss_score', 'kev_flag', 'soft_label', 'xgb_score']]
    save_dataframe(top20_xgb, 'top20_ltr_recommendations', subdir='evaluation')
    print("[OK] Top-20 XGBoost recommendations saved (LambdaMART not run)")
else:
    print("[WARN] No LTR or XGBoost scores available -- skipping Top-20 save")


TOP-20 RECOMMENDATIONS ANALYSIS


XGBoost:
  High-priority (label≥2): 20/20 (100.0%)
  KEV flags: 20/20 (100.0%)
  Mean CVSS: 9.17
  Mean confidence: 1.000

LambdaMART:
  High-priority (label≥2): 20/20 (100.0%)
  KEV flags: 20/20 (100.0%)
  Mean CVSS: 7.21
  Mean confidence: 1.000

Heuristic:
  High-priority (label≥2): 20/20 (100.0%)
  KEV flags: 20/20 (100.0%)
  Mean CVSS: 9.50
  Mean confidence: 1.000

CVSS:
  High-priority (label≥2): 3/20 (15.0%)
  KEV flags: 3/20 (15.0%)
  Mean CVSS: 10.00
  Mean confidence: 0.468

[OK] DataFrame saved: outputs/evaluation/top20_ltr_recommendations.csv
[OK] Top-20 LTR recommendations saved


## 15. Model Export — All Artifacts

Persist trained models and ensemble metadata to disk so STEP_5 can run reproducible evaluation on fixed artifacts.

In [18]:
# Save ALL trained model artifacts
print_header('MODEL EXPORT — ALL ARTIFACTS')

model_path          = project_root / 'models' / 'ltr_ranker.model'
xgb_model_path      = project_root / 'models' / 'xgb_ranker.model'
model_path.parent.mkdir(parents=True, exist_ok=True)

# LambdaMART (standard variant)
if 'ltr_model' in globals():
    save_model(ltr_model, str(model_path))
    register_artifact('ltr_ranker_model', model_path)
    print(f"[OK] LambdaMART (standard)  → {model_path.name}  ({model_path.stat().st_size/1024:.1f} KB)")
else:
    print("[WARN] LambdaMART not trained -- skipping ltr_ranker.model export")

# XGBoost
if 'xgb_model' in globals():
    xgb_model.save_model(str(xgb_model_path))
    register_artifact('xgb_ranker_model', xgb_model_path)
    print(f"[OK] XGBoost Ranker         → {xgb_model_path.name}  ({xgb_model_path.stat().st_size/1024:.1f} KB)")
else:
    print("[WARN] XGBoost not trained -- skipping xgb_ranker.model export")

# DiffusionRank: stateless algorithm — seed model is LambdaMART; no separate file needed
print("[OK] DiffusionRank          → scores derived from ltr_ranker.model (stateless re-ranking)")

# RGCN: save if trained successfully
rgcn_path = project_root / 'models' / 'rgcn_simple.pt'
import torch
try:
    torch.save(rgcn_model.state_dict(), str(rgcn_path))
    register_artifact('rgcn_model', rgcn_path)
    print(f"[OK] RGCN                   → {rgcn_path.name}  ({rgcn_path.stat().st_size/1024:.1f} KB)")
except Exception as e:
    print(f"[WARN] RGCN save failed: {e}")

# Ensemble: stateless (weights computed at inference); save weights dict
import json as _json
ens_weights_path = project_root / 'models' / 'ensemble_weights.json'
try:
    _json.dump(val_ndcg_weights, open(str(ens_weights_path), 'w'), indent=2)
    register_artifact('ensemble_weights', ens_weights_path)
    print(f"[OK] Ensemble weights       → {ens_weights_path.name}")
except Exception as e:
    print(f"[WARN] Ensemble weights save failed: {e}")

print_header('MODELS READY FOR STEP 5 EVALUATION')
print(f"  ltr_ranker.model         — LambdaMART (primary recommended model)")
print(f"  ltr_ranker_thesis_70_30  — LambdaMART (explicit 70/30 holdout split variant)")
print(f"  xgb_ranker.model         — XGBoost Ranker")
print(f"  rgcn_simple.pt           — RGCN (2-layer GCN)")
print(f"  ensemble_weights.json    — Ensemble fusion weights")
print(f"\n  All test-set scores stored in: outputs/evaluation/model_comparison_test_results.csv")

trace_stage_done(
    'model_export',
    status='ok',
    models_dir=str((project_root / 'models').resolve())
)


MODEL EXPORT — ALL ARTIFACTS

[OK] LambdaMART (standard)  → ltr_ranker.model  (57.9 KB)
[OK] XGBoost Ranker         → xgb_ranker.model  (270.2 KB)
[OK] DiffusionRank          → scores derived from ltr_ranker.model (stateless re-ranking)
[OK] RGCN                   → rgcn_simple.pt  (42.3 KB)
[OK] Ensemble weights       → ensemble_weights.json

MODELS READY FOR STEP 5 EVALUATION

  ltr_ranker.model         — LambdaMART (primary recommended model)
  ltr_ranker_thesis_70_30  — LambdaMART (explicit 70/30 holdout split variant)
  xgb_ranker.model         — XGBoost Ranker
  rgcn_simple.pt           — RGCN (2-layer GCN)
  ensemble_weights.json    — Ensemble fusion weights

  All test-set scores stored in: outputs/evaluation/model_comparison_test_results.csv
[TRACE] model_export | ok | {'models_dir': '/Users/vinayksharma/AirDnd/cti_recommender/models'}


## 16. Training Summary

Summarize dataset scale, main model outcomes, and reference-relative gains for a concise end-of-training report.



In [19]:
print_header('MODEL TRAINING & EVALUATION COMPLETE')

print()
print('[STATS] Dataset:')
print(f"  Total CVEs: {len(df):,}")
print(f"  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"  Features: {len(feature_cols)}")

print()
print('[TARGET] Primary Model: Learning-to-Rank: LambdaMART')
if 'ltr_model' in globals():
    print(f"  Training iterations: {ltr_model.best_iteration}")
    print(f"  Best validation NDCG@10: {ltr_model.best_score['valid']['ndcg@10']:.4f}")
else:
    print("  [WARN] LambdaMART not trained in this session")

print()
print(' Test Set Performance:')
if 'results' in dir() and 'Learning-to-Rank: LambdaMART (Ours)' in results:
    ltr_metrics = results['Learning-to-Rank: LambdaMART (Ours)']
    print(f"  NDCG@10: {ltr_metrics.get('NDCG@10', 0):.4f}")
    print(f"  NDCG@20: {ltr_metrics.get('NDCG@20', 0):.4f}")
    print(f"  Precision@10: {ltr_metrics.get('Precision@10', 0):.4f}")
    print(f"  MAP: {ltr_metrics.get('MAP', 0):.4f}")

    print()
    print(' Relative gain vs CVSS Reference:')
    cvss_metrics = results['CVSS Reference']
    for metric in ['NDCG@10', 'NDCG@20', 'Precision@10']:
        if metric in ltr_metrics and metric in cvss_metrics and cvss_metrics[metric] > 0:
            improvement = ((ltr_metrics[metric] - cvss_metrics[metric]) / cvss_metrics[metric]) * 100
            print(f"  {metric}: {improvement:+.1f}%")
else:
    print('  [WARN] Run Model Comparison cell first to see detailed metrics')

print()
print('[STATS] Statistical Significance:')
if 'significance_df' in dir() and not significance_df.empty:
    for _, r in significance_df.iterrows():
        flag = '[OK]' if r['Significant_alpha_0_05'] else '[INFO]'
        print(f"  {r['Comparison']}: p={r['p_value']:.6f} {flag}")
else:
    print('  [WARN] Run Statistical Significance cell first')

print()
print(' Outputs:')
print('  Model: models/ltr_ranker.model')
print('  Results: outputs/evaluation/')
print('  Plots: outputs/plots/')

print()
print_separator()
print('\n[OK] Model ready for production use')
print('[OK] Run scripts/evaluation/recommend_cves.py to generate recommendations')


MODEL TRAINING & EVALUATION COMPLETE


[STATS] Dataset:
  Total CVEs: 210,147
  Train: 147,102 | Val: 31,522 | Test: 31,523
  Features: 47

[TARGET] Primary Model: Learning-to-Rank: LambdaMART
  Training iterations: 10
  Best validation NDCG@10: 1.0000

 Test Set Performance:
  [WARN] Run Model Comparison cell first to see detailed metrics

[STATS] Statistical Significance:
  Learning-to-Rank: LambdaMART vs CVSS Reference: p=0.000000 [OK]
  Learning-to-Rank: LambdaMART vs Heuristic Reference: p=0.000001 [OK]
  Learning-to-Rank: LambdaMART vs Learning-to-Rank: XGBoost: p=0.885823 [INFO]

 Outputs:
  Model: models/ltr_ranker.model
  Results: outputs/evaluation/
  Plots: outputs/plots/



[OK] Model ready for production use
[OK] Run scripts/evaluation/recommend_cves.py to generate recommendations


## 17. Unified Scientific Protocol (Artifact-Driven)

Run a single reproducible protocol and consume its standardized artifacts for thesis-grade evaluation.

In [20]:
# Create primary temporal holdout split
print_header('PRIMARY TEMPORAL HOLDOUT EVALUATION')

cutoff_date = THESIS_CUTOFF_DATE

df_thesis_train = df[df['published'] <= cutoff_date].copy()
df_thesis_test = df[df['published'] > cutoff_date].copy()

print(f"\n Split Strategy:")
print(f"  Train: Published ≤ {cutoff_date.date()}")
print(f"  Test:  Published > {cutoff_date.date()}")

print(f"\n[STATS] Split Sizes:")
print(f"  Train: {len(df_thesis_train):,} CVEs ({len(df_thesis_train)/len(df)*100:.1f}%)")
print(f"  Test:  {len(df_thesis_test):,} CVEs ({len(df_thesis_test)/len(df)*100:.1f}%)")

print(f"\n Date Ranges:")
print(f"  Train: {df_thesis_train['published'].min().date()} to {df_thesis_train['published'].max().date()}")
print(f"  Test:  {df_thesis_test['published'].min().date()} to {df_thesis_test['published'].max().date()}")

print(f"\n  Label Distribution:")
print(f"  Train high-priority (label≥2): {(df_thesis_train['soft_label'] >= 2).mean()*100:.1f}%")
print(f"  Test high-priority (label≥2):  {(df_thesis_test['soft_label'] >= 2).mean()*100:.1f}%")

print_separator()


PRIMARY TEMPORAL HOLDOUT EVALUATION


 Split Strategy:
  Train: Published ≤ 2024-12-31
  Test:  Published > 2024-12-31

[STATS] Split Sizes:
  Train: 165,683 CVEs (78.8%)
  Test:  44,464 CVEs (21.2%)

 Date Ranges:
  Train: 2018-01-01 to 2024-12-30
  Test:  2024-12-31 to 2025-12-31

  Label Distribution:
  Train high-priority (label≥2): 3.3%
  Test high-priority (label≥2):  0.8%



In [21]:
# Prepare primary holdout train/test data (add published_week for grouping)
df_thesis_train['published_week'] = df_thesis_train['published'].dt.tz_localize(None).dt.to_period('W').astype(str)
df_thesis_test['published_week'] = df_thesis_test['published'].dt.tz_localize(None).dt.to_period('W').astype(str)

# Create validation split from training window only (no test leakage)
df_thesis_train = df_thesis_train.sort_values('published').copy()
val_size = max(1, int(len(df_thesis_train) * 0.15))
if val_size >= len(df_thesis_train):
    val_size = max(1, len(df_thesis_train) // 5)

split_idx = len(df_thesis_train) - val_size
df_thesis_train_fit = df_thesis_train.iloc[:split_idx].copy()
df_thesis_val = df_thesis_train.iloc[split_idx:].copy()

cat_mapping_thesis = fit_categorical_mapping(df_thesis_train_fit, CATEGORICAL_COLS)
df_thesis_train_fit = apply_categorical_mapping(df_thesis_train_fit, cat_mapping_thesis)
df_thesis_val = apply_categorical_mapping(df_thesis_val, cat_mapping_thesis)
df_thesis_test_enc = apply_categorical_mapping(df_thesis_test, cat_mapping_thesis)

y_thesis_train = df_thesis_train_fit['soft_label']
y_thesis_val = df_thesis_val['soft_label']
y_thesis_test = df_thesis_test_enc['soft_label']

print('Primary holdout training data (leakage-safe):')
print(f"  Train-fit: {len(df_thesis_train_fit):,} CVEs")
print(f"  Val:       {len(df_thesis_val):,} CVEs")
print(f"  Test:      {len(df_thesis_test_enc):,} CVEs")
print(f"  Train-fit period: {df_thesis_train_fit['published'].min().date()} to {df_thesis_train_fit['published'].max().date()}")
print(f"  Val period:       {df_thesis_val['published'].min().date()} to {df_thesis_val['published'].max().date()}")
print(f"  Test period:      {df_thesis_test_enc['published'].min().date()} to {df_thesis_test_enc['published'].max().date()}")



Primary holdout training data (leakage-safe):
  Train-fit: 140,831 CVEs
  Val:       24,852 CVEs
  Test:      44,464 CVEs
  Train-fit period: 2018-01-01 to 2024-05-14
  Val period:       2024-05-14 to 2024-12-30
  Test period:      2024-12-31 to 2025-12-31


In [22]:
# Train LambdaMART on primary temporal holdout split
print('')
print('Training LambdaMART on primary holdout split (train/val from pre-cutoff only)...')

# Allow this cell to run even if Section 17 has not been executed yet.
ltr_params_thesis = ltr_params if 'ltr_params' in globals() else get_default_ltr_params()

ltr_model_thesis = train_lambdarank(
    df_thesis_train_fit,
    df_thesis_val,
    feature_cols,
    params=ltr_params_thesis,
    random_seed=42
)

print('[OK] Training complete')
print(f"  Best iteration: {ltr_model_thesis.best_iteration}")
print(f"  Best validation NDCG@10: {ltr_model_thesis.best_score['valid']['ndcg@10']:.4f}")

X_thesis_test_array = df_thesis_test_enc[feature_cols].fillna(0).values
ltr_scores_thesis_test = ltr_model_thesis.predict(X_thesis_test_array)
cvss_scores_thesis_test = compute_cvss_only_scores(df_thesis_test_enc)
heuristic_scores_thesis_test = compute_heuristic_scores(df_thesis_test_enc)



Training LambdaMART on primary holdout split (train/val from pre-cutoff only)...

TRAINING CONFIDENCE-WEIGHTED LAMBDARANK

Feature diagnostics (train):
  Total configured features: 47
  Zero-variance features: 0
  Mostly-zero (<1% non-zero): 7
  [INFO] Mostly-zero: ['kev_flag', 'is_healthcare', 'is_curated', 'curated_severity', 'desc_has_xxe', 'vendor_is_healthcare', 'healthcare_critical']

Feature diagnostics (validation):
  Total configured features: 47
  Zero-variance features: 2
  Mostly-zero (<1% non-zero): 7
  [WARN] Zero-variance: ['is_curated', 'curated_severity']
  [INFO] Mostly-zero: ['kev_flag', 'is_healthcare', 'is_curated', 'curated_severity', 'cwe_is_crypto', 'desc_has_xxe', 'healthcare_critical']

Preparing training data...
  Train: 140,831 samples, 333 groups
  Confidence weights: min=0.200, mean=0.350, max=1.000

Preparing validation data...
  Val: 24,852 samples, 34 groups

Training LambdaRank model...
Parameters: {'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_e

In [23]:
# Evaluate on primary temporal holdout test set
print_header('PRIMARY HOLDOUT EVALUATION RESULTS')

models_thesis = {
    'CVSS Reference (Primary Holdout)': cvss_scores_thesis_test,
    'Heuristic Reference (Primary Holdout)': heuristic_scores_thesis_test,
    'Learning-to-Rank: LambdaMART (Primary Holdout)': ltr_scores_thesis_test
}

results_thesis = {}
for model_name, scores in models_thesis.items():
    metrics = compute_ranking_metrics_inline(y_thesis_test, scores, k_values=EVAL_K_VALUES)
    results_thesis[model_name] = metrics

print('\nPrimary Holdout Test Set Metrics:')
for model_name, metrics in results_thesis.items():
    print(f"\n{model_name}:")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.4f}")

results_thesis_df = pd.DataFrame(results_thesis).T
save_dataframe(results_thesis_df, 'thesis_70_30_evaluation_results', subdir='evaluation')
print('\n[OK] Primary holdout evaluation results saved')



PRIMARY HOLDOUT EVALUATION RESULTS


Primary Holdout Test Set Metrics:

CVSS Reference (Primary Holdout):
  Precision@10: 0.0000
  Recall@10: 0.0000
  NDCG@10: 0.2961
  Precision@20: 0.1500
  Recall@20: 0.0082
  NDCG@20: 0.3243
  Precision@100: 0.1300
  Recall@100: 0.0355
  NDCG@100: 0.3664
  MAP: 0.0402

Heuristic Reference (Primary Holdout):
  Precision@10: 1.0000
  Recall@10: 0.0273
  NDCG@10: 0.7550
  Precision@20: 1.0000
  Recall@20: 0.0546
  NDCG@20: 0.8269
  Precision@100: 0.9700
  Recall@100: 0.2650
  NDCG@100: 0.9270
  MAP: 0.7539

Learning-to-Rank: LambdaMART (Primary Holdout):
  Precision@10: 1.0000
  Recall@10: 0.0273
  NDCG@10: 0.9321
  Precision@20: 1.0000
  Recall@20: 0.0546
  NDCG@20: 0.9661
  Precision@100: 1.0000
  Recall@100: 0.2732
  NDCG@100: 0.9871
  MAP: 0.9999
[OK] DataFrame saved: outputs/evaluation/thesis_70_30_evaluation_results.csv

[OK] Primary holdout evaluation results saved


## 18. LambdaMART — Variant 1: Standard Split (70 / 15 / 15)

Confidence-weighted LambdaRank with NDCG@10 objective on the standard 70/15/15 split. This is a comparison variant; the primary model is trained in Section 11.

**LambdaMART Variants Summary**:
| Variant | Training split | Purpose |
|---------|---------------|----------|
| V1 (Section 18) | 70/15/15 temporal | Standard split comparison |
| V2 (Section 19) | 70/30 random holdout | Explicit percentage-split comparison |
| V3 (Section 20) | 5-fold grouped K-Fold | Robustness / stability |


In [24]:
# Train LambdaMART with functional API
print_header('TRAINING CONFIDENCE-WEIGHTED LAMBDAMART')
print(f"\n  Model Configuration:")
print(f"  Objective: LambdaRank (pairwise ranking)")
print(f"  Metric: NDCG (Normalized Discounted Cumulative Gain)")
print(f"  Trees: 500 (with early stopping)")
print(f"  Max depth: 6")
print(f"  Learning rate: 0.05")
print(f"  Confidence weighting: Enabled")

# Add published_week column for grouping (remove timezone before period conversion)
train_df['published_week'] = train_df['published'].dt.tz_localize(None).dt.to_period('W').astype(str)
val_df['published_week'] = val_df['published'].dt.tz_localize(None).dt.to_period('W').astype(str)
test_df['published_week'] = test_df['published'].dt.tz_localize(None).dt.to_period('W').astype(str)

# Verify all feature columns are numeric before training
print(f"\n[DEBUG] Verifying feature dtypes before training:")
non_numeric_features = []
for col in feature_cols:
    if col in train_df.columns:
        dtype = train_df[col].dtype
        if dtype == 'object':
            non_numeric_features.append(f"{col} (dtype: {dtype})")
            print(f"  [ERROR] {col}: {dtype} - Sample: {train_df[col].iloc[0]}")
        elif dtype.name not in ['int8', 'int16', 'int32', 'int64', 'float16', 'float32', 'float64', 'bool']:
            print(f"  [WARN] {col}: {dtype}")

if non_numeric_features:
    raise ValueError(f"Non-numeric features detected: {non_numeric_features}")
else:
    print(f"  [OK] All {len(feature_cols)} features are numeric")

# Train model
print(f"\nTraining...")
ltr_params_v1 = get_default_ltr_params()
ltr_model_v1_standard = train_lambdarank(train_df, val_df, feature_cols, params=ltr_params_v1, random_seed=42)

print(f"\n[OK] Training complete")
print(f"  Best iteration: {ltr_model_v1_standard.best_iteration}")
print(f"  Best validation NDCG@10: {ltr_model_v1_standard.best_score['valid']['ndcg@10']:.4f}")
print_separator()

# Generate predictions
X_train_array = train_df[feature_cols].fillna(0).values
X_val_array = val_df[feature_cols].fillna(0).values
X_test_array = test_df[feature_cols].fillna(0).values
ltr_scores_v1_standard_val = ltr_model_v1_standard.predict(X_val_array)
ltr_scores_v1_standard_test = ltr_model_v1_standard.predict(X_test_array)

trace_stage_done(
    'lambdamart_train',
    status='ok',
    best_iteration=int(ltr_model_v1_standard.best_iteration),
    best_valid_ndcg10=float(ltr_model_v1_standard.best_score['valid']['ndcg@10']),
    n_features=int(len(feature_cols))
)


TRAINING CONFIDENCE-WEIGHTED LAMBDAMART


  Model Configuration:
  Objective: LambdaRank (pairwise ranking)
  Metric: NDCG (Normalized Discounted Cumulative Gain)
  Trees: 500 (with early stopping)
  Max depth: 6
  Learning rate: 0.05
  Confidence weighting: Enabled

[DEBUG] Verifying feature dtypes before training:
  [OK] All 47 features are numeric

Training...

TRAINING CONFIDENCE-WEIGHTED LAMBDARANK

Feature diagnostics (train):
  Total configured features: 47
  Zero-variance features: 0
  Mostly-zero (<1% non-zero): 7
  [INFO] Mostly-zero: ['kev_flag', 'is_healthcare', 'is_curated', 'curated_severity', 'desc_has_xxe', 'vendor_is_healthcare', 'healthcare_critical']

Feature diagnostics (validation):
  Total configured features: 47
  Zero-variance features: 2
  Mostly-zero (<1% non-zero): 6
  [WARN] Zero-variance: ['is_curated', 'curated_severity']
  [INFO] Mostly-zero: ['kev_flag', 'is_curated', 'curated_severity', 'cwe_is_crypto', 'desc_has_xxe', 'healthcare_critical']

Preparing

## 19. LambdaMART Variant 2: Explicit Percentage Split (70/30)

Train LambdaMART using an explicit 70%/30% split created via train_test_split, with validation carved from the 70% training subset only.



In [25]:
# Build and evaluate explicit 70/30 holdout variant
from sklearn.model_selection import train_test_split

print_header('LAMBDARANK VARIANT 2: EXPLICIT 70/30 HOLDOUT')

df_70_30_train, df_70_30_test = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

df_70_30_train = df_70_30_train.copy()
df_70_30_test = df_70_30_test.copy()
df_70_30_train['published_week'] = df_70_30_train['published'].dt.tz_localize(None).dt.to_period('W').astype(str)
df_70_30_test['published_week'] = df_70_30_test['published'].dt.tz_localize(None).dt.to_period('W').astype(str)

df_70_30_train = df_70_30_train.sort_values('published').copy()
val_size_70_30 = max(1, int(len(df_70_30_train) * 0.15))
if val_size_70_30 >= len(df_70_30_train):
    val_size_70_30 = max(1, len(df_70_30_train) // 5)
split_idx_70_30 = len(df_70_30_train) - val_size_70_30
df_70_30_train_fit = df_70_30_train.iloc[:split_idx_70_30].copy()
df_70_30_val = df_70_30_train.iloc[split_idx_70_30:].copy()

cat_mapping_70_30 = fit_categorical_mapping(df_70_30_train_fit, CATEGORICAL_COLS)
df_70_30_train_fit = apply_categorical_mapping(df_70_30_train_fit, cat_mapping_70_30)
df_70_30_val = apply_categorical_mapping(df_70_30_val, cat_mapping_70_30)
df_70_30_test_enc = apply_categorical_mapping(df_70_30_test, cat_mapping_70_30)

y_70_30_test = df_70_30_test_enc['soft_label']

print('[STATS] 70/30 split sizes:')
print(f"  Train-fit: {len(df_70_30_train_fit):,} CVEs")
print(f"  Val:       {len(df_70_30_val):,} CVEs")
print(f"  Test:      {len(df_70_30_test_enc):,} CVEs")

ltr_params_70_30 = ltr_params if 'ltr_params' in globals() else get_default_ltr_params()
ltr_model_70_30 = train_lambdarank(
    df_70_30_train_fit,
    df_70_30_val,
    feature_cols,
    params=ltr_params_70_30,
    random_seed=42
)

X_70_30_test_array = df_70_30_test_enc[feature_cols].fillna(0).values
ltr_scores_70_30_test = ltr_model_70_30.predict(X_70_30_test_array)
cvss_scores_70_30_test = compute_cvss_only_scores(df_70_30_test_enc)
heuristic_scores_70_30_test = compute_heuristic_scores(df_70_30_test_enc)

models_thesis = {
    'CVSS Reference (70/30 Holdout)': cvss_scores_70_30_test,
    'Heuristic Reference (70/30 Holdout)': heuristic_scores_70_30_test,
    'Learning-to-Rank: LambdaMART (70/30 Holdout)': ltr_scores_70_30_test
}

results_thesis = {}
for model_name, scores in models_thesis.items():
    results_thesis[model_name] = compute_ranking_metrics_inline(y_70_30_test, scores, k_values=EVAL_K_VALUES)

print('\n70/30 Holdout Test Set Metrics:')
for model_name, metrics in results_thesis.items():
    print(f"\n{model_name}:")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.4f}")

results_thesis_df = pd.DataFrame(results_thesis).T
save_dataframe(results_thesis_df, 'thesis_70_30_evaluation_results', subdir='evaluation')
print('\n[OK] 70/30 holdout evaluation results saved')

comparison_data = []
results_main = results if 'results' in globals() else {}
main_ltr_key = None
for candidate in ['LambdaMART (LTR)', 'Learning-to-Rank: LambdaMART (Ours)', 'Learning-to-Rank: LambdaMART']:
    if candidate in results_main:
        main_ltr_key = candidate
        break

if main_ltr_key is not None:
    for metric_name, value in results_main[main_ltr_key].items():
        comparison_data.append({'Split Strategy': 'Standard 70/15/15', 'Metric': metric_name, 'Score': value})

for metric_name, value in results_thesis['Learning-to-Rank: LambdaMART (70/30 Holdout)'].items():
    comparison_data.append({'Split Strategy': 'Explicit 70/30 Holdout', 'Metric': metric_name, 'Score': value})

comparison_df = pd.DataFrame(comparison_data)
if len(comparison_df) > 0:
    comparison_pivot = comparison_df.pivot(index='Metric', columns='Split Strategy', values='Score')
    if 'Standard 70/15/15' in comparison_pivot.columns and 'Explicit 70/30 Holdout' in comparison_pivot.columns:
        comparison_pivot['Difference'] = comparison_pivot['Explicit 70/30 Holdout'] - comparison_pivot['Standard 70/15/15']
        base_vals = comparison_pivot['Standard 70/15/15'].replace(0, np.nan)
        comparison_pivot['% Change'] = (comparison_pivot['Difference'] / base_vals) * 100
    print('\nLambdaMART Performance Comparison:')
    print(comparison_pivot.to_string())

    fig = px.bar(
        comparison_df[comparison_df['Metric'].isin(['NDCG@10', 'NDCG@20', 'Precision@10', 'Precision@20'])],
        x='Metric',
        y='Score',
        color='Split Strategy',
        barmode='group',
        title='LambdaMART: Standard 70/15/15 vs Explicit 70/30 Holdout',
        labels={'Score': 'Score', 'Metric': ''},
        text='Score'
    )
    fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
    fig.update_layout(height=450, yaxis_range=[0, 1.1])
    save_plot(fig, 'original_vs_thesis_split_comparison')
    print('\n[OK] Comparison plot saved')


LAMBDARANK VARIANT 2: EXPLICIT 70/30 HOLDOUT

[STATS] 70/30 split sizes:
  Train-fit: 125,037 CVEs
  Val:       22,065 CVEs
  Test:      63,045 CVEs

TRAINING CONFIDENCE-WEIGHTED LAMBDARANK

Feature diagnostics (train):
  Total configured features: 47
  Zero-variance features: 0
  Mostly-zero (<1% non-zero): 6
  [INFO] Mostly-zero: ['kev_flag', 'is_healthcare', 'is_curated', 'curated_severity', 'desc_has_xxe', 'healthcare_critical']

Feature diagnostics (validation):
  Total configured features: 47
  Zero-variance features: 2
  Mostly-zero (<1% non-zero): 6
  [WARN] Zero-variance: ['is_curated', 'curated_severity']
  [INFO] Mostly-zero: ['kev_flag', 'is_curated', 'curated_severity', 'cwe_is_crypto', 'desc_has_xxe', 'healthcare_critical']

Preparing training data...
  Train: 125,037 samples, 380 groups
  Confidence weights: min=0.200, mean=0.341, max=1.000

Preparing validation data...
  Val: 22,065 samples, 39 groups

Training LambdaRank model...
Parameters: {'objective': 'lambdarank'

In [26]:
# Save 70/30 holdout model separately
model_path_thesis = project_root / 'models' / 'ltr_ranker_thesis_70_30.model'
save_model(ltr_model_70_30, str(model_path_thesis))

print_header('70/30 HOLDOUT MODEL EXPORT')
print(f"\n[OK] 70/30 holdout model saved to: {model_path_thesis.name}")
print(f"  Training data: Random 70% split")
print(f"  Tested on: Random 30% split")
print(f"  Size: {model_path_thesis.stat().st_size / 1024:.1f} KB")
print(f"\n[OK] Primary model: ltr_ranker.model (recommended temporal holdout protocol)")
print(f"[OK] Variant model: ltr_ranker_thesis_70_30.model (explicit 70/30 holdout split)")
print(f"\nBoth models available for comparison and deployment")
print_separator()


70/30 HOLDOUT MODEL EXPORT


[OK] 70/30 holdout model saved to: ltr_ranker_thesis_70_30.model
  Training data: Random 70% split
  Tested on: Random 30% split
  Size: 9.7 KB

[OK] Primary model: ltr_ranker.model (recommended temporal holdout protocol)
[OK] Variant model: ltr_ranker_thesis_70_30.model (explicit 70/30 holdout split)

Both models available for comparison and deployment



## 20. LambdaMART Variant 3: Grouped K-Fold Cross-Validation

**Purpose**: Evaluate using grouped K-Fold as per supervisor requirement
- Split data into K folds grouped by `published_week`
- Train on K-1 folds, test on 1 fold
- Repeat K times, average results
- Compare with temporal splits to show robustness

**Advantage over single split evaluation**:
- Uses all data efficiently
- Reduces variance in estimates
- Preserves group separation across train/test
- Standard ML evaluation practice


In [27]:
trace_event('kfold_setup', status='start')
from sklearn.model_selection import GroupKFold
import warnings

print_header('K-FOLD CROSS VALIDATION (GROUPED BY PUBLISHED WEEK)')

n_folds = 5
random_state = 42

print()
print('  Configuration:')
print(f"  Folds: {n_folds}")
print('  Strategy: GroupKFold')
print('  Grouping key: published_week')
print(f"  Random state: {random_state}")

df_kfold = df.copy()
df_kfold['published_week'] = df_kfold['published'].dt.tz_localize(None).dt.to_period('W').astype(str)
X_kfold = df_kfold[feature_cols].fillna(0)
y_kfold = df_kfold['soft_label']
groups_kfold = df_kfold['published_week']

print()
print('[STATS] Data:')
print(f"  Total CVEs: {len(df_kfold):,}")
print(f"  Features: {len(feature_cols)}")
print(f"  Unique weekly groups: {groups_kfold.nunique():,}")
print(f"  Label distribution: {y_kfold.value_counts().sort_index().to_dict()}")

skf = GroupKFold(n_splits=n_folds)

print()
print_separator()
print(f"Starting {n_folds}-Fold Cross Validation...")
print_separator()
print()
trace_event('kfold_setup', status='ok', folds=n_folds, n_rows=len(df_kfold))

[TRACE] kfold_setup | start | 

K-FOLD CROSS VALIDATION (GROUPED BY PUBLISHED WEEK)


  Configuration:
  Folds: 5
  Strategy: GroupKFold
  Grouping key: published_week
  Random state: 42

[STATS] Data:
  Total CVEs: 210,147
  Features: 47
  Unique weekly groups: 418
  Label distribution: {0: 105328, 1: 99068, 2: 5704, 3: 47}


Starting 5-Fold Cross Validation...


[TRACE] kfold_setup | ok | {'folds': 5, 'n_rows': 210147}


In [28]:
trace_event('kfold_run', status='start')
from src.evaluation.metrics import evaluate_ranking

kfold_results = []
fold_models = []
ltr_params_kfold = ltr_params if 'ltr_params' in globals() else get_default_ltr_params()

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_kfold, y_kfold, groups=groups_kfold), 1):
    trace_event('kfold_fold', status='start', fold=fold_idx)
    print()
    print_separator(char='-')
    print(f"Fold {fold_idx}/{n_folds}")
    print_separator(char='-')

    df_fold_train_full = df_kfold.iloc[train_idx].copy().sort_values('published')
    df_fold_test = df_kfold.iloc[test_idx].copy()

    fold_val_size = max(1, int(len(df_fold_train_full) * 0.15))
    if fold_val_size >= len(df_fold_train_full):
        fold_val_size = max(1, len(df_fold_train_full) // 5)

    fold_split_idx = len(df_fold_train_full) - fold_val_size
    df_fold_train = df_fold_train_full.iloc[:fold_split_idx].copy()
    df_fold_val = df_fold_train_full.iloc[fold_split_idx:].copy()

    fold_cat_map = fit_categorical_mapping(df_fold_train, CATEGORICAL_COLS)
    df_fold_train = apply_categorical_mapping(df_fold_train, fold_cat_map)
    df_fold_val = apply_categorical_mapping(df_fold_val, fold_cat_map)
    df_fold_test = apply_categorical_mapping(df_fold_test, fold_cat_map)

    y_fold_test = df_fold_test['soft_label']

    print(f"  Train-fit: {len(df_fold_train):,} CVEs")
    print(f"  Val:       {len(df_fold_val):,} CVEs")
    print(f"  Test:      {len(df_fold_test):,} CVEs")
    print(f"  Test date range: {df_fold_test['published'].min().date()} to {df_fold_test['published'].max().date()}")

    print('  Training LambdaMART...')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        fold_model = train_lambdarank(
            df_fold_train,
            df_fold_val,
            feature_cols,
            params=ltr_params_kfold,
            random_seed=random_state + fold_idx
        )
    fold_models.append(fold_model)

    X_fold_test_array = df_fold_test[feature_cols].fillna(0).values
    ltr_scores_fold = fold_model.predict(X_fold_test_array)
    cvss_scores_fold = compute_cvss_only_scores(df_fold_test)
    heuristic_scores_fold = compute_heuristic_scores(df_fold_test)

    models_fold = {
        'CVSS Reference': cvss_scores_fold,
        'Heuristic Reference': heuristic_scores_fold,
        'Learning-to-Rank: LambdaMART': ltr_scores_fold
    }

    last_ltr_ndcg10 = float('nan')
    for model_name, scores in models_fold.items():
        inline_m = compute_ranking_metrics_inline(y_fold_test, scores, k_values=EVAL_K_VALUES)

        _df_eval = df_fold_test.copy()
        _df_eval['_kfold_score'] = scores
        grouped_m = evaluate_ranking(_df_eval, '_kfold_score', label_col='soft_label', group_col='published_week', k_values=EVAL_K_VALUES)

        metrics = {}
        for k in EVAL_K_VALUES:
            metrics[f'NDCG@{k}'] = grouped_m[f'NDCG@{k}']
            metrics[f'Precision@{k}'] = inline_m[f'Precision@{k}']
            metrics[f'Recall@{k}'] = inline_m.get(f'Recall@{k}', 0.0)
        metrics['MAP'] = inline_m['MAP']
        metrics['Fold'] = fold_idx
        metrics['Model'] = model_name
        kfold_results.append(metrics)

        if model_name == 'Learning-to-Rank: LambdaMART':
            last_ltr_ndcg10 = grouped_m.get('NDCG@10', float('nan'))

    print(f"  [OK] Fold {fold_idx} complete - Grouped NDCG@10: {last_ltr_ndcg10:.4f}")
    trace_event('kfold_fold', status='ok', fold=fold_idx, ndcg10=float(last_ltr_ndcg10))

print_header('K-Fold Cross Validation Complete')
trace_event('kfold_run', status='ok', total_folds=n_folds, rows=len(kfold_results))

[TRACE] kfold_run | start | 
[TRACE] kfold_fold | start | {'fold': 1}


----------------------------------------------------------------------
Fold 1/5

----------------------------------------------------------------------
  Train-fit: 142,952 CVEs
  Val:       25,226 CVEs
  Test:      41,969 CVEs
  Test date range: 2018-01-22 to 2025-12-31
  Training LambdaMART...

TRAINING CONFIDENCE-WEIGHTED LAMBDARANK

Feature diagnostics (train):
  Total configured features: 47
  Zero-variance features: 0
  Mostly-zero (<1% non-zero): 6
  [INFO] Mostly-zero: ['kev_flag', 'is_healthcare', 'is_curated', 'curated_severity', 'desc_has_xxe', 'healthcare_critical']

Feature diagnostics (validation):
  Total configured features: 47
  Zero-variance features: 2
  Mostly-zero (<1% non-zero): 6
  [WARN] Zero-variance: ['is_curated', 'curated_severity']
  [INFO] Mostly-zero: ['kev_flag', 'is_curated', 'curated_severity', 'cwe_is_crypto', 'desc_has_xxe', 'healthcare_critical']

Preparing training data...
  Tr

In [29]:
# Aggregate K-Fold results
kfold_df = pd.DataFrame(kfold_results)

print_header('K-FOLD AGGREGATED RESULTS')

agg_cols = [c for c in kfold_df.columns if c not in ('Fold', 'Model')]
agg_dict = {col: ['mean', 'std'] for col in agg_cols}

kfold_summary = kfold_df.groupby('Model').agg(agg_dict).round(4)

model_order = ['CVSS Reference', 'Heuristic Reference', 'Learning-to-Rank: LambdaMART']
print('\nK-Fold Results (Mean +/- Std across 5 folds):')
for model in model_order:
    if model not in kfold_summary.index:
        continue
    print(f"\n{model}:")
    for k in EVAL_K_VALUES:
        for prefix in ['NDCG', 'Precision']:
            metric = f'{prefix}@{k}'
            if (metric, 'mean') in kfold_summary.columns:
                mean_val = kfold_summary.loc[model, (metric, 'mean')]
                std_val = kfold_summary.loc[model, (metric, 'std')]
                print(f"  {metric}: {mean_val:.4f} +/- {std_val:.4f}")
    if ('MAP', 'mean') in kfold_summary.columns:
        mean_val = kfold_summary.loc[model, ('MAP', 'mean')]
        std_val = kfold_summary.loc[model, ('MAP', 'std')]
        print(f"  MAP: {mean_val:.4f} +/- {std_val:.4f}")

save_dataframe(kfold_df, 'kfold_all_results', subdir='evaluation')
save_dataframe(kfold_summary, 'kfold_summary_results', subdir='evaluation')

print('\n[OK] K-Fold results saved')
print_separator()
print()


K-FOLD AGGREGATED RESULTS


K-Fold Results (Mean +/- Std across 5 folds):

CVSS Reference:
  NDCG@10: 0.2955 +/- 0.0161
  Precision@10: 0.0800 +/- 0.0447
  NDCG@20: 0.3333 +/- 0.0099
  Precision@20: 0.1300 +/- 0.0447
  NDCG@100: 0.4673 +/- 0.0109
  Precision@100: 0.1460 +/- 0.0114
  MAP: 0.0558 +/- 0.0032

Heuristic Reference:
  NDCG@10: 0.6966 +/- 0.0104
  Precision@10: 1.0000 +/- 0.0000
  NDCG@20: 0.7276 +/- 0.0101
  Precision@20: 1.0000 +/- 0.0000
  NDCG@100: 0.7580 +/- 0.0042
  Precision@100: 1.0000 +/- 0.0000
  MAP: 0.4764 +/- 0.0143

Learning-to-Rank: LambdaMART:
  NDCG@10: 0.9902 +/- 0.0016
  Precision@10: 1.0000 +/- 0.0000
  NDCG@20: 0.9934 +/- 0.0008
  Precision@20: 1.0000 +/- 0.0000
  NDCG@100: 0.9957 +/- 0.0005
  Precision@100: 1.0000 +/- 0.0000
  MAP: 0.9998 +/- 0.0003
[OK] DataFrame saved: outputs/evaluation/kfold_all_results.csv
[OK] DataFrame saved: outputs/evaluation/kfold_summary_results.csv

[OK] K-Fold results saved




In [30]:
# Visualize K-Fold results
metrics_to_plot = ['NDCG@10', 'NDCG@20', 'Precision@10', 'Precision@20']

# Per-fold comparison
fold_plot_data = kfold_df[kfold_df['Model'] == 'Learning-to-Rank: LambdaMART'].copy()
fold_plot_data = fold_plot_data[['Fold'] + metrics_to_plot].melt(id_vars='Fold', var_name='Metric', value_name='Score')

fig = px.line(
    fold_plot_data,
    x='Fold',
    y='Score',
    color='Metric',
    markers=True,
    title='Learning-to-Rank: LambdaMART Performance Across K-Folds',
    labels={'Score': 'Score', 'Fold': 'Fold Number'},
)
fig.update_layout(height=400)
save_plot(fig, 'kfold_performance_per_fold')

# Mean comparison across models
mean_comparison = []
model_order = ['CVSS Reference', 'Heuristic Reference', 'Learning-to-Rank: LambdaMART']
for model in model_order:
    if model not in kfold_summary.index:
        continue
    for metric in metrics_to_plot:
        mean_comparison.append({
            'Model': model,
            'Metric': metric,
            'Score': kfold_summary.loc[model, (metric, 'mean')],
            'Std': kfold_summary.loc[model, (metric, 'std')]
        })

mean_comparison_df = pd.DataFrame(mean_comparison)

fig = px.bar(
    mean_comparison_df,
    x='Metric',
    y='Score',
    color='Model',
    barmode='group',
    error_y='Std',
    title='K-Fold Cross Validation: Mean Performance +/- Std',
    labels={'Score': 'Score', 'Metric': ''},
    text='Score'
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(height=450, yaxis_range=[0, 1.1])
save_plot(fig, 'kfold_mean_comparison')

print('[OK] K-Fold visualization saved')



[OK] K-Fold visualization saved


## 21. Feature Importance & SHAP Analysis

Explainability for the primary LambdaMART model — highlights which CVE features (EPSS, CVSS score, exploit age, CWE) drive ranking decisions.

In [31]:
# Get feature importance from LightGBM
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': ltr_model.feature_importance(importance_type='gain')
}).sort_values('Importance', ascending=False)

# Filter to only non-zero importance features
feature_importance_nonzero = feature_importance[feature_importance['Importance'] > 0]
num_used_features = len(feature_importance_nonzero)
num_unused_features = len(feature_cols) - num_used_features

print_header('FEATURE IMPORTANCE (LightGBM Gain)')
print(f"\n[STATS] Feature Usage:")
print(f"  Features with non-zero importance: {num_used_features}/{len(feature_cols)}")
print(f"  Unused features (importance = 0): {num_unused_features}")

print(f"\nTop {min(15, num_used_features)} Most Important Features:")
for i, row in feature_importance_nonzero.head(15).iterrows():
    bar = '█' * int(row['Importance'] / feature_importance['Importance'].max() * 30)
    print(f"  {row['Feature']:30s}: {bar} {row['Importance']:.0f}")

if num_unused_features > 0:
    print(f"\n[NOTE] {num_unused_features} features unused by model (zero importance)")
    unused_features = feature_importance[feature_importance['Importance'] == 0]['Feature'].tolist()
    print(f"  Examples: {', '.join(unused_features[:5])}")
    if num_unused_features > 5:
        print(f"  ... and {num_unused_features - 5} more")

print_separator()

# Visualize feature importance (only non-zero)
num_to_plot = min(15, num_used_features)
fig = px.bar(
    feature_importance_nonzero.head(num_to_plot),
    x='Importance',
    y='Feature',
    orientation='h',
    title=f'Top {num_to_plot} Features by Importance (LightGBM Gain)',
    labels={'Importance': 'Gain', 'Feature': ''},
    color='Importance',
    color_continuous_scale='Viridis'
)
fig.update_layout(height=max(300, num_to_plot * 30), showlegend=False)

save_plot(fig, 'feature_importance_ltr')
print("[OK] Feature importance plot saved")


FEATURE IMPORTANCE (LightGBM Gain)


[STATS] Feature Usage:
  Features with non-zero importance: 32/47
  Unused features (importance = 0): 15

Top 15 Most Important Features:
  epss_percentile               : ██████████████████████████████ 6345
  kev_flag                      : █████████████████ 3624
  attack_flag                   : ██ 631
  epss_score                    : ██ 476
  desc_has_xss                  :  187
  cvss_score_derived            :  176
  vendor_is_high_risk           :  101
  cvss_av                       :  68
  vendor_risk_score             :  5
  chpl_flag                     :  3
  cwe_is_injection              :  3
  desc_has_dos                  :  3
  attack_technique_count        :  1
  cwe_severity_score            :  1
  cvss_pr                       :  1

[NOTE] 15 features unused by model (zero importance)
  Examples: high_impact_network, auth_not_required, network_accessible, vendor_is_healthcare, desc_has_xxe
  ... and 10 more

[OK] Feature importan

## 22. Final Comparison: All Three LambdaMART Training Variants

Compare performance across all three evaluation approaches:
1. **Original 70/15/15**: Standard temporal split
2. **70/30 Holdout**: Explicit percentage split built with train_test_split
3. **K-Fold CV**: Grouped-by-week folds (5-fold; avoids same-week leakage across train/test)


In [32]:
# Compare all three evaluation strategies
print_header('COMPREHENSIVE COMPARISON: ALL EVALUATION STRATEGIES')

all_strategies_data = []

if 'Learning-to-Rank: LambdaMART (Ours)' in results:
    for metric_name, value in results['Learning-to-Rank: LambdaMART (Ours)'].items():
        all_strategies_data.append({
            'Strategy': 'Original (70/15/15)',
            'Metric': metric_name,
            'Score': value,
            'Type': 'Single Test'
        })

if 'Learning-to-Rank: LambdaMART (70/30 Holdout)' in results_thesis:
    for metric_name, value in results_thesis['Learning-to-Rank: LambdaMART (70/30 Holdout)'].items():
        all_strategies_data.append({
            'Strategy': 'Explicit 70/30 Holdout',
            'Metric': metric_name,
            'Score': value,
            'Type': 'Single Test'
        })

kfold_metric_prefixes = ['NDCG', 'Precision', 'Recall']
kfold_metrics = [f'{prefix}@{k}' for prefix in kfold_metric_prefixes for k in EVAL_K_VALUES] + ['MAP']
for metric in kfold_metrics:
    if (metric, 'mean') in kfold_summary.columns:
        mean_val = kfold_summary.loc['Learning-to-Rank: LambdaMART', (metric, 'mean')]
        std_val = kfold_summary.loc['Learning-to-Rank: LambdaMART', (metric, 'std')]
        all_strategies_data.append({
            'Strategy': f'K-Fold CV (n={n_folds})',
            'Metric': metric,
            'Score': mean_val,
            'Std': std_val,
            'Type': 'Cross-Validation'
        })

all_strategies_df = pd.DataFrame(all_strategies_data)

key_metrics = [f'NDCG@{k}' for k in EVAL_K_VALUES if f'NDCG@{k}' in all_strategies_df['Metric'].values]
key_metrics += [f'Precision@{k}' for k in EVAL_K_VALUES if f'Precision@{k}' in all_strategies_df['Metric'].values]
if 'MAP' in all_strategies_df['Metric'].values:
    key_metrics.append('MAP')
comparison_table = all_strategies_df[all_strategies_df['Metric'].isin(key_metrics)].pivot(
    index='Metric',
    columns='Strategy',
    values='Score'
)

print('\nLambdaMART Performance Across Evaluation Strategies:')
print(comparison_table.to_string())

kfold_stds = all_strategies_df[
    (all_strategies_df['Strategy'] == f'K-Fold CV (n={n_folds})') &
    (all_strategies_df['Metric'].isin(key_metrics))
][['Metric', 'Std']].set_index('Metric')

print('\n\nK-Fold Standard Deviations:')
for metric in key_metrics:
    if metric in kfold_stds.index:
        print(f"  {metric}: +/-{kfold_stds.loc[metric, 'Std']:.4f}")

save_dataframe(all_strategies_df, 'all_evaluation_strategies_comparison', subdir='evaluation')
print('\n[OK] Comprehensive comparison saved')
print_separator()


COMPREHENSIVE COMPARISON: ALL EVALUATION STRATEGIES


LambdaMART Performance Across Evaluation Strategies:
Strategy       Explicit 70/30 Holdout  K-Fold CV (n=5)
Metric                                                
MAP                          0.999468           0.9998
NDCG@10                      0.685185           0.9902
NDCG@100                     0.911613           0.9957
NDCG@20                      0.765008           0.9934
Precision@10                 1.000000           1.0000
Precision@100                1.000000           1.0000
Precision@20                 1.000000           1.0000


K-Fold Standard Deviations:
  NDCG@10: +/-0.0016
  NDCG@20: +/-0.0008
  NDCG@100: +/-0.0005
  Precision@10: +/-0.0000
  Precision@20: +/-0.0000
  Precision@100: +/-0.0000
  MAP: +/-0.0003
[OK] DataFrame saved: outputs/evaluation/all_evaluation_strategies_comparison.csv

[OK] Comprehensive comparison saved



In [33]:
# Visualize comprehensive comparison
metrics_to_compare = ['NDCG@10', 'NDCG@20', 'Precision@10', 'Precision@20']

plot_data = all_strategies_df[all_strategies_df['Metric'].isin(metrics_to_compare)].copy()

fig = px.bar(
    plot_data,
    x='Metric',
    y='Score',
    color='Strategy',
    barmode='group',
    title='LambdaMART: Comprehensive Evaluation Strategy Comparison',
    labels={'Score': 'Score', 'Metric': ''},
    text='Score',
    error_y='Std' if 'Std' in plot_data.columns else None
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(height=500, yaxis_range=[0, 1.1])

save_plot(fig, 'comprehensive_strategy_comparison')

print_header('KEY INSIGHTS')
print(f"\n1. Complete data split (70/15/15):")
print(f"   - Standard temporal split")
print(f"   - Single test set evaluation")
print(f"   - Good for quick validation")

print(f"\n2. Explicit 70/30 Holdout:")
print(f"   - Built with random train/test split (70%/30%)")
print(f"   - Clean percentage-based benchmark")
print(f"   - Useful as a complementary validation strategy")

print(f"\n3. K-Fold CV (n={n_folds}):")
print(f"   - Folding mechanism as requested")
print(f"   - Grouped by published_week to avoid group leakage")
print(f"   - Provides confidence intervals (mean ± std)")
print(f"   - More robust estimate of model performance")
print(f"   - Uses all data efficiently")

print(f"\n[STATS] Recommendation:")
print(f"   - K-Fold shows robust performance with confidence intervals")
print(f"   - Explicit 70/30 holdout provides percentage-split validation")
print(f"   - Complete data split validates temporal ordering")
print(f"\n[OK] Comprehensive evaluation plot saved")
print_separator()


KEY INSIGHTS


1. Complete data split (70/15/15):
   - Standard temporal split
   - Single test set evaluation
   - Good for quick validation

2. Explicit 70/30 Holdout:
   - Built with random train/test split (70%/30%)
   - Clean percentage-based benchmark
   - Useful as a complementary validation strategy

3. K-Fold CV (n=5):
   - Folding mechanism as requested
   - Grouped by published_week to avoid group leakage
   - Provides confidence intervals (mean ± std)
   - More robust estimate of model performance
   - Uses all data efficiently

[STATS] Recommendation:
   - K-Fold shows robust performance with confidence intervals
   - Explicit 70/30 holdout provides percentage-split validation
   - Complete data split validates temporal ordering

[OK] Comprehensive evaluation plot saved



## 23. Hyperparameter Sensitivity and Fallback Audit

This section adds a compact LambdaMART sensitivity sweep and explicit post-training fallback diagnostics for graph models.

In [34]:
import numpy as np
import pandas as pd

print_header('HYPERPARAMETER SENSITIVITY (COMPACT SWEEP) + FALLBACK AUDIT')

try:
    sweep_ok = all(k in globals() for k in ['train_lambdarank', 'compute_ranking_metrics_inline', 'feature_cols', 'train_df', 'val_df'])
    sensitivity_rows = []

    if sweep_ok:
        sample_frac = 0.35
        _train = train_df.sample(frac=sample_frac, random_state=42) if len(train_df) > 5000 else train_df.copy()
        _val = val_df.sample(frac=sample_frac, random_state=42) if len(val_df) > 5000 else val_df.copy()

        sweep_grid = [
            {'learning_rate': 0.03, 'num_leaves': 31, 'min_data_in_leaf': 20},
            {'learning_rate': 0.05, 'num_leaves': 31, 'min_data_in_leaf': 20},
            {'learning_rate': 0.05, 'num_leaves': 63, 'min_data_in_leaf': 30},
            {'learning_rate': 0.08, 'num_leaves': 63, 'min_data_in_leaf': 30},
        ]

        for i, params_delta in enumerate(sweep_grid, 1):
            base_params = dict(ltr_params) if 'ltr_params' in globals() else {}
            base_params.update(params_delta)

            model_i = train_lambdarank(_train, _val, feature_cols, params=base_params, random_seed=100 + i)
            yv = _val['soft_label']
            sv = model_i.predict(_val[feature_cols].fillna(0).values)
            mv = compute_ranking_metrics_inline(yv, sv, k_values=[10, 20])

            sensitivity_rows.append({
                'run_id': i,
                'learning_rate': base_params.get('learning_rate'),
                'num_leaves': base_params.get('num_leaves'),
                'min_data_in_leaf': base_params.get('min_data_in_leaf'),
                'NDCG@10': float(mv.get('NDCG@10', np.nan)),
                'NDCG@20': float(mv.get('NDCG@20', np.nan)),
                'MAP': float(mv.get('MAP', np.nan)),
            })

        ltr_sensitivity_df = pd.DataFrame(sensitivity_rows).sort_values('NDCG@10', ascending=False)
        print('\nTop sweep results:')
        print(ltr_sensitivity_df.to_string(index=False))

        out_path = project_root / 'outputs' / 'evaluation' / 'lambdamart_hparam_sensitivity.csv'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        ltr_sensitivity_df.to_csv(out_path, index=False)
        print(f"\n[OK] Saved hyperparameter sensitivity -> {out_path}")
    else:
        print('[WARN] Skipped sweep (missing training objects in kernel).')

    print('\nFallback audit:')
    if 'test_df' in globals() and isinstance(test_df, pd.DataFrame):
        if 'rgcn_score' in test_df.columns:
            rgcn_sum = float(np.nansum(test_df['rgcn_score'].values))
            rgcn_nonzero = int((np.abs(test_df['rgcn_score'].values) > 1e-12).sum())
            print(f"  RGCN non-zero predictions: {rgcn_nonzero:,} / {len(test_df):,} | sum={rgcn_sum:.6f}")
            if rgcn_nonzero == 0:
                print('  [WARN] RGCN appears to have fully fallback/zero output.')
        else:
            print('  [WARN] Missing `rgcn_score` in test_df.')

        if 'diffusion_score' in test_df.columns:
            diff_nonzero = int((np.abs(test_df['diffusion_score'].values) > 1e-12).sum())
            print(f"  Diffusion non-zero predictions: {diff_nonzero:,} / {len(test_df):,}")
        else:
            print('  [WARN] Missing `diffusion_score` in test_df.')
    else:
        print('  [WARN] test_df not available for fallback audit.')

except Exception as e:
    print(f"[ERROR] Sensitivity/fallback audit failed: {e}")


HYPERPARAMETER SENSITIVITY (COMPACT SWEEP) + FALLBACK AUDIT


TRAINING CONFIDENCE-WEIGHTED LAMBDARANK

Feature diagnostics (train):
  Total configured features: 47
  Zero-variance features: 0
  Mostly-zero (<1% non-zero): 7
  [INFO] Mostly-zero: ['kev_flag', 'is_healthcare', 'is_curated', 'curated_severity', 'desc_has_xxe', 'vendor_is_healthcare', 'healthcare_critical']

Feature diagnostics (validation):
  Total configured features: 47
  Zero-variance features: 2
  Mostly-zero (<1% non-zero): 7
  [WARN] Zero-variance: ['is_curated', 'curated_severity']
  [INFO] Mostly-zero: ['kev_flag', 'is_curated', 'curated_severity', 'cwe_is_crypto', 'desc_has_auth_bypass', 'desc_has_xxe', 'healthcare_critical']

Preparing training data...
  Train: 51,486 samples, 341 groups
  Confidence weights: min=0.200, mean=0.349, max=1.000

Preparing validation data...
  Val: 11,033 samples, 40 groups

Training LambdaRank model...
Parameters: {'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [5, 1

## Next Steps

1. **Generate Recommendations** -> Run `scripts/evaluation/recommend_cves.py` to score new CVEs
2. **Monitor Performance** -> Track model performance over time
3. **Retrain Periodically** -> Update model with new data quarterly
4. **Evaluate Drift** -> Check for distribution shift in new CVEs

---